# События мониторинга ПКБ в софт-коллекшн: влияние и информационная ценность (IV)

**Вопрос.** Какие события мониторинга ПКБ (Первое кредитное бюро) несут сигнал для софт-коллекшн (ранняя стадия
взыскания, просрочка 1–30 дней) и какой прирост качества (инкрементальный эффект) они дают сверх внутренних данных,
которые уже использует стратегия взыскания?

**Входные данные** (те же объекты, что и в рабочем ноутбуке)

| объект | гранулярность | используемые столбцы |
|---|---|---|
| `ovrd_contr` | одна строка на договор и `act_date` с просрочкой (`ovd_days_cnt` > 0), объединённая с атрибутами договора | `contract_number`, `iin`, `act_date`, `ovd_days_cnt`, `process_ref`, `open_date`, `close_date`, `end_date`, `status_name`, `contract_amount`, `partner_name`, `product_id` |
| `hes` | одна строка на событие мониторинга ПКБ | `bin_iin`, `rule_name`, `old_value`, `new_value` (текст JSON), `created_date`, `event_date`, `internal_event_id` |
| `contracts` *(необязательно)* | одна строка на договор, выданный 2025-01-01 … 2026-10-15 | атрибуты договора; при наличии таблицы выполняется раздел раннего предупреждения (часть B) |

**Методика (на момент скоринга, без заглядывания в будущее)**
1. **Эпизоды просрочки** восстанавливаются по строкам с днями просрочки (DPD): `dlq_start = act_date − (ovd_days_cnt − 1)`.
   Это работает независимо от того, есть ли в `ovrd_contr` строка за каждый день просрочки или только за некоторые дни месяца.
2. **Популяция (выборка), часть A**: каждый выход договора в софт-коллекшн (скоринг выполняется в день, когда эпизод
   достигает `OBS_DPD`, по умолчанию DPD 1), если дата скоринга попадает в период покрытия мониторингом, а исход
   эпизода полностью наблюдаем.
3. **Целевая переменная (таргет)**: `bad = 1` («плохой»), если эпизод достигает DPD ≥ 31 (перекат из софт- в хард-коллекшн).
   Дополнительный таргет: договор не вышел из просрочки (нет самоизлечения) к DPD 16.
4. **Признаки**: события ПКБ, поступившие в банк **до** даты скоринга (`created_date` < даты скоринга),
   агрегированные за 7 / 14 / 30 дней по каждому правилу, плюс суммы и изменения, извлечённые из JSON. События по
   собственным договорам банка (`contract_code` совпадает с нашим `contract_number` или `process_ref`) вынесены в
   отдельную группу, потому что они повторяют информацию, которая у банка уже есть.
5. **Анализ**: WoE (вес свидетельства) / IV с монотонным разбиением на бины (интервалы), Джини (Gini) и KS по каждому
   признаку, лифт событий по каждому правилу, понедельная стабильность и PSI, фильтр по корреляции, а также прирост
   Джини модели относительно базовой модели на внутренних данных на выборке вне времени (out-of-time, OOT)
   с доверительным интервалом (ДИ), полученным бутстрэпом.
6. **Часть B (необязательно)**: те же признаки для договоров, которые пока обслуживаются без просрочки; прогнозируется
   выход на просрочку в течение 30 дней — раннее предупреждение (EWS) до начала софт-коллекшн.

**Подписи на русском языке.** Все таблицы показываются и выгружаются в Excel с русскими названиями колонок и
значений (словари `RU_COLUMNS` и `RU_VALUES` и функция `ru_labels` в первой ячейке кода). Внутри кода имена колонок
и коды (например, `pkb_other`, `iv`, `bad_rate`) остаются английскими, поэтому логика расчёта не меняется.

**Как запустить на реальных данных**: выполните эти ячейки в ядре, где уже загружены `ovrd_contr` и `hes` (и `contracts`,
если он у вас есть), или вставьте эти ячейки в рабочий ноутбук. По умолчанию `DEMO_MODE = False`; имена ваших датафреймов
задаются в `OVRD_VAR`, `HES_VAR`, `CONTRACTS_VAR`. Раздел 2b проверяет датафреймы до любого тяжёлого шага и
останавливается с понятным сообщением, если что-то не подходит.
При `DEMO_MODE = True` ноутбук строит синтетический набор данных с теми же столбцами и структурой JSON, чтобы каждую
ячейку можно было проверить от начала до конца. **Результаты на синтетических демо-данных ничего не говорят о реальном портфеле.**

In [ ]:
import os, re, gc, json, math, time, uuid, warnings
from datetime import date
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

try:                                   # если установлен orjson, разбор JSON в 3–5 раз быстрее
    import orjson
    _json_loads = orjson.loads
except ImportError:
    _json_loads = json.loads

try:
    display                            # noqa: F821  (функция display есть внутри Jupyter)
except NameError:
    display = print                    # вне Jupyter таблицы просто печатаются

# ------------------------------------------------------------------ русские подписи таблиц (слой отображения)
# Внутренние имена колонок и коды значений в коде остаются английскими: на них опирается логика расчёта.
# Переводятся только подписи при показе таблиц (display) и при выгрузке в Excel/CSV: ru_labels() возвращает
# переименованную КОПИЮ таблицы, числа и исходные объекты не меняются.

# имена колонок и индексов -> русские подписи
RU_COLUMNS = {
    # исходные таблицы (просмотр демо-данных)
    'rep_month': 'Отчётный месяц', 'act_date': 'Дата среза', 'contract_number': 'Номер договора',
    'ovd_days_cnt': 'Дни просрочки (DPD)', 'process_ref': 'Референс процесса', 'iin': 'ИИН',
    'open_date': 'Дата открытия', 'close_date': 'Дата закрытия', 'end_date': 'Плановая дата окончания',
    'status_name': 'Статус договора', 'contract_amount': 'Сумма договора', 'partner_name': 'Партнёр',
    'product_id': 'Код продукта', 'date_value': 'Дата значения', 'date_change': 'Дата изменения',
    'internal_event_id': 'ID события', 'bin_iin': 'БИН/ИИН', 'rule_name': 'Название правила',
    'old_value': 'Старое значение (JSON)', 'new_value': 'Новое значение (JSON)', 'event_date': 'Дата события',
    'created_date': 'Дата поступления в банк',
    # предварительная проверка
    'object': 'Объект', 'check': 'Проверка', 'status': 'Статус', 'detail': 'Подробности',
    # атрибуты договоров
    'contract': 'Договор', 'amount': 'Сумма', 'product': 'Продукт', 'channel': 'Канал', 'closed': 'Закрыт',
    'from_contracts': 'Из таблицы contracts',
    # качество данных, объёмы событий, когорты, сопоставление с нашими договорами
    'value': 'Значение', 'description': 'Описание', 'rule': 'Правило', 'count': 'Число событий в выборке',
    '50%': 'Медиана, ч', '90%': '90-й перцентиль, ч', '99%': '99-й перцентиль, ч',
    'week': 'Неделя скоринга (начало)', 'episodes': 'Число эпизодов', 'bads': 'Число «плохих»',
    'bad_rate': 'Доля «плохих»', 'events': 'Число событий',
    'share_with_contract_code': 'Доля событий с кодом договора', 'share_own_contract': 'Доля событий по нашим договорам',
    # каталог признаков, бины, IV
    'feature': 'Признак', 'group': 'Группа признаков', 'family': 'Семейство правил',
    'window': 'Окно наблюдения, дн.', 'bin': 'Бин (интервал)', 'n': 'Число наблюдений', 'share': 'Доля бина',
    'woe': 'WoE', 'iv': 'IV', 'strength': 'Сила признака по IV', 'gini': 'Джини', 'ks': 'KS',
    'n_bins': 'Число бинов', 'share_nonzero': 'Доля ненулевых значений', 'share_missing': 'Доля пропусков',
    'bads_outside_main_bin': 'Число «плохих» вне основного бина', 'reliable': 'Надёжен',
    'direction': 'Направление зависимости',
    # влияние правил (лифт)
    'rule_key': 'Код правила', 'share_flagged': 'Доля наблюдений с событием', 'n_flagged': 'Число наблюдений с событием',
    'bads_flagged': 'Число «плохих» с событием', 'bad_rate_flagged': 'Доля «плохих» с событием',
    'bad_rate_not': 'Доля «плохих» без события', 'lift': 'Лифт', 'iv_flag': 'IV флага события',
    # понедельная стабильность
    'iv_min': 'IV мин. по неделям', 'iv_max': 'IV макс. по неделям', 'iv_cv': 'Коэф. вариации IV',
    'gini_min_week': 'Джини мин. по неделям', 'gini_max_week': 'Джини макс. по неделям', 'psi_max': 'PSI макс.',
    'stable': 'Стабилен', 'iv_all_episodes': 'IV (все эпизоды)',
    'iv_full_30d_history': 'IV (окно наблюдения заполнено полностью)',
    # прирост качества
    'model': 'Модель', 'n_features': 'Число признаков', 'gini_train': 'Джини, обучающая выборка',
    'gini_test': 'Джини, тестовая выборка (OOT)', 'capture_top10%': 'Доля выявленных «плохих» в топ-10%',
    'capture_top20%': 'Доля выявленных «плохих» в топ-20%', 'capture_top30%': 'Доля выявленных «плохих» в топ-30%',
    'delta_gini_vs_A': 'Прирост Джини к модели A', 'delta_CI95': '95% ДИ прироста Джини',
    'gini_test_boosting': 'Джини бустинга, тестовая выборка', 'delta_boosting_CI95': '95% ДИ прироста Джини (бустинг)',
    'coef': 'Коэффициент', 'sign_ok': 'Знак ожидаемый (отрицательный)',
}

# коды значений (группы, семейства, сила IV, подписи бинов, статусы, названия моделей) -> русские подписи
RU_VALUES = {
    # группы признаков
    'pkb_other': 'ПКБ: другие кредиторы', 'pkb_bank_rule': 'ПКБ: правила относительно нашего банка',
    'pkb_own': 'ПКБ: наши договоры', 'baseline': 'базовые (внутренние) признаки',
    # семейства правил и служебные значения каталога признаков
    'credit_hunger': 'кредитный голод', 'relief': 'облегчение условий (пролонгация, реструктуризация)',
    'delinquency': 'просрочка', 'positive': 'позитивные события', 'status': 'особые статусы клиента',
    'technical': 'технические события', 'debt_burden': 'долговая нагрузка',
    'bank_rule': 'правила относительно нашего банка', 'other': 'прочие', 'unknown': 'неизвестные правила',
    'all': 'все', 'family': 'все правила семейства',
    # сила признака по IV и направление зависимости риска от значения
    'not useful': 'бесполезный', 'weak': 'слабый', 'medium': 'средний', 'strong': 'сильный',
    'suspicious: check leakage': 'подозрительно высокая IV: проверить утечку данных',
    'risk up with value': 'риск растёт при увеличении значения', 'risk down with value': 'риск снижается при увеличении значения',
    # подписи бинов и значения канала продаж
    'MISSING': 'пропуск', 'OTHER': 'прочие значения',
    'direct': 'прямой', 'merchant': 'через партнёра (мерчант)', 'marketplace': 'маркетплейс', 'NA': 'нет данных',
    # статусы предварительной проверки (OK не переводится)
    'WARN': 'ПРЕДУПРЕЖДЕНИЕ', 'FAIL': 'ОШИБКА',
    # модели в таблице прироста качества
    'A: baseline': 'A: базовая модель (внутренние признаки)',
    'B: + PKB other lenders': 'B: + ПКБ (другие кредиторы и правила банка)',
    'C: + PKB own contracts': 'C: + ПКБ по нашим договорам',
}


def _ru_value(v):
    # одно значение: перевод из RU_VALUES, если это строка из словаря; числа, даты, bool и прочее не меняются
    return RU_VALUES.get(v, v) if isinstance(v, str) else v


def _ru_column(c):
    # имя колонки или индекса: RU_COLUMNS, затем имена с параметрами (недели, целевые переменные), затем RU_VALUES
    if not isinstance(c, str):
        return c
    if c in RU_COLUMNS:
        return RU_COLUMNS[c]
    m = re.fullmatch(r'iv_(\d\d\.\d\d)', c)                     # понедельная IV: iv_<дд.мм>
    if m:
        return f'IV, неделя с {m.group(1)}'
    if c == f'iv_roll{BAD_DPD}':                                # IV по основной целевой переменной
        return f'IV: перекат в DPD {BAD_DPD}+ (основной таргет)'
    for name, thr in EXTRA_TARGETS.items():                     # дополнительные целевые переменные
        if c == f'iv_{name}':
            return f'IV: DPD ≥ {thr} (дополнительный таргет)'
        if c == f'{name}_rate':
            return f'Доля эпизодов с DPD ≥ {thr}'
    return RU_VALUES.get(c, c)


def _is_text(dtype):
    # текстовые колонки: object, string/str и category (числа, bool и даты не трогаются)
    return (dtype == object or isinstance(dtype, pd.CategoricalDtype)
            or (pd.api.types.is_string_dtype(dtype) and not pd.api.types.is_bool_dtype(dtype)))


def _ru_index(idx, values):
    # Index: имена уровней переводятся как имена колонок; метки — как значения (values=True) или как имена колонок
    if _is_text(idx.dtype):
        idx = idx.map(_ru_value if values else _ru_column)
    return idx.set_names([_ru_column(nm) for nm in idx.names])


def ru_labels(obj):
    """Копия таблицы pandas с русскими подписями: имена колонок и индексов — по RU_COLUMNS, текстовые значения и
    метки индекса — по RU_VALUES. Числовые данные не меняются, исходный объект не изменяется;
    объекты других типов (текст, числа, графики) возвращаются как есть."""
    if isinstance(obj, pd.Series):
        out = obj.astype(object).map(_ru_value) if _is_text(obj.dtype) else obj.copy()
        out.index = _ru_index(obj.index, values=True)
        out.name = _ru_column(obj.name)
        return out
    if not isinstance(obj, pd.DataFrame):
        return obj
    out = obj.copy()
    cols, out.columns = obj.columns, range(obj.shape[1])         # временно номера колонок (имена могут повторяться)
    for j in range(obj.shape[1]):
        if _is_text(out[j].dtype):
            out[j] = out[j].astype(object).map(_ru_value)
    # если у оси колонок есть имя (crosstab, unstack), её метки — это значения (например, сила IV), иначе — имена колонок
    out.columns = _ru_index(cols, values=cols.name is not None)
    out.index = _ru_index(obj.index, values=True)
    return out


# display показывает каждую таблицу через ru_labels (работает и с запасным вариантом display = print)
if not getattr(display, '_ru_labels', False):                  # при повторном запуске ячейки не оборачиваем дважды
    _display_orig = display


def display(*objs, **kw):
    return _display_orig(*[ru_labels(o) for o in objs], **kw)


display._ru_labels = True

# скрываем технические предупреждения pandas/numpy и задаём формат вывода таблиц
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 220)
pd.set_option('display.max_colwidth', 70)
pd.set_option('display.float_format', lambda v: f'{v:,.4f}')

# ------------------------------------------------------------------ режим запуска
DEMO_MODE = False           # False: ваши датафреймы уже загружены в этом ядре. True: синтетические демо-данные.
SEED = 42                   # зерно генератора случайных чисел (воспроизводимость демо-данных, выборок и бутстрэпа)

# ------------------------------------------------------------------ имена ВАШИХ датафреймов в этом ядре
OVRD_VAR = 'ovrd_contr'     # строки с просрочкой, объединённые с атрибутами договоров
HES_VAR = 'hes'             # события мониторинга ПКБ (Первое кредитное бюро)
CONTRACTS_VAR = 'contracts' # необязательно: все выданные договоры (нужны для части B); если такого объекта нет, часть B просто пропускается

# ------------------------------------------------------------------ имена столбцов (измените, если у вас они другие)
# столбцы наших данных (ovrd_contr / contracts): договор, ИИН, дата среза, дни просрочки (DPD)
C_CONTRACT, C_IIN, C_ACT_DATE, C_DPD = 'contract_number', 'iin', 'act_date', 'ovd_days_cnt'
# process_ref (ещё один номер договора, тоже сверяется с contract_code из ПКБ), даты открытия, закрытия и окончания договора
C_PROCESS_REF, C_OPEN, C_CLOSE, C_END = 'process_ref', 'open_date', 'close_date', 'end_date'
# статус договора, сумма договора, партнёр, продукт
C_STATUS, C_AMOUNT, C_PARTNER, C_PRODUCT = 'status_name', 'contract_amount', 'partner_name', 'product_id'
# столбцы таблицы событий ПКБ (hes): ИИН/БИН клиента, название правила, старое и новое значение (JSON)
H_IIN, H_RULE, H_OLD, H_NEW = 'bin_iin', 'rule_name', 'old_value', 'new_value'
# идентификатор события, дата события в ПКБ, дата поступления события в банк
H_ID, H_EVENT_DATE, H_CREATED = 'internal_event_id', 'event_date', 'created_date'

# ------------------------------------------------------------------ логика «на момент скоринга» (без заглядывания в будущее)
EVENT_TIME_COL = H_CREATED  # когда событие поступило в банк (безопасный вариант). event_date = когда оно произошло в ПКБ.
EVENT_LAG_DAYS = 0          # сколько дней ещё выждать, прежде чем событие можно использовать (запас на задержку в промышленной среде)
OBS_DPD = 1                 # точка скоринга: день, когда эпизод достигает этого DPD (1 = первый день софт-коллекшн)
BAD_DPD = 31                # основная целевая переменная (таргет): эпизод достигает DPD >= 31
EXTRA_TARGETS = {'nocure_16': 16}   # дополнительные таргеты: имя -> порог DPD (добавьте 'roll_61': 61, когда истории станет достаточно)
WINDOWS = (7, 14, 30)       # окна наблюдения в днях
IMPACT_WINDOW = 14          # окно для таблицы лифта по правилам
MIN_HISTORY_DAYS = 14       # первая дата скоринга = начало мониторинга + столько дней
HES_START = None            # начало мониторинга; None = определить по дневным объёмам, либо дата, например '2026-09-07'
DATA_END = None             # последний день с полными данными по DPD; None = max(act_date)
EPISODE_START_TOL_DAYS = 1  # допустимый разброс dlq_start (в днях) внутри одного эпизода
BANK_RULES = ('partial_ovd', 'refin_out_80')   # правила ПКБ, заданные относительно нашего банка

# ------------------------------------------------------------------ биннинг, отбор признаков, модель
MAX_BINS = 6                # максимальное число бинов (интервалов) на признак
MIN_BIN_SHARE = 0.05        # минимальная доля наблюдений в бине; более мелкие бины объединяются с соседними
MIN_BADS_RELIABLE = 100     # IV помечается как ненадёжная, если вне основного бина меньше стольких «плохих»
IV_SELECT = 0.02            # минимальная IV, чтобы признак ПКБ стал кандидатом в модель
CORR_MAX = 0.75             # максимальная |корреляция Спирмена| с уже отобранным признаком
MAX_MODEL_FEATURES = 15     # максимальное число отбираемых признаков ПКБ в модели
TRAIN_SHARE = 0.70          # доля обучающей выборки; разбиение вне времени (OOT) по дате скоринга
N_BOOT = 200                # число итераций бутстрэпа для доверительного интервала (ДИ) прироста Джини
RUN_HGB = True              # контрольная проверка прироста качества на градиентном бустинге
RUN_HES_DQ = True           # проверка качества данных по всей таблице hes (понедельные количества по правилам, задержки)
DQ_SAMPLE = 2_000_000       # сколько строк hes брать в выборку для статистики задержек и предварительных проверок
STOP_ON_PREFLIGHT_FAIL = True   # True: остановить ноутбук, если предварительная проверка дала статус ОШИБКА (FAIL)

# ------------------------------------------------------------------ часть B: раннее предупреждение (EWS) (нужен `contracts`)
RUN_EWS = True              # выполнять часть B (только если загружена таблица contracts)
EWS_HORIZON = 30            # договор без просрочки выходит на просрочку в течение стольких дней ...
EWS_MIN_DPD = 1             # ... и этот эпизод достигает как минимум такого DPD
EWS_SNAPSHOT_FREQ_DAYS = 7  # шаг (в днях) между датами среза для популяции B
EWS_MAX_ROWS = 400_000      # максимум строк в популяции B; при превышении берётся случайная подвыборка

OUTPUT_DIR = 'pkb_iv_output'   # папка для сохранения результатов (Excel или CSV)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# стиль графиков: тонкие линии, тонкая сетка, фиксированный порядок категорий
# PAL: палитра цветов (ключи - внутренние имена цветов, не менять)
PAL = {'blue': '#2a78d6', 'orange': '#eb6834', 'aqua': '#1baf7a', 'yellow': '#eda100',
       'magenta': '#e87ba4', 'green': '#008300', 'violet': '#6250d6', 'red': '#e34948',
       'gray': '#8c8b85', 'ink': '#0b0b0b', 'muted': '#52514e'}
SERIES = [PAL[k] for k in ('blue', 'orange', 'aqua', 'yellow', 'magenta', 'green', 'violet', 'red')]   # порядок цветов линий
# цвет группы признаков: pkb_other - другие кредиторы, pkb_bank_rule - правила относительно нашего банка,
# pkb_own - наши договоры, baseline - базовые (внутренние) признаки
GROUP_COLOR = {'pkb_other': PAL['blue'], 'pkb_bank_rule': PAL['orange'], 'pkb_own': PAL['aqua'], 'baseline': PAL['yellow']}
plt.rcParams.update({
    'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True,
    'grid.color': '#e6e5e1', 'grid.linewidth': 0.6, 'axes.edgecolor': '#b5b4ae', 'axes.linewidth': 0.6,
    'axes.titlesize': 10.5, 'axes.titleweight': 'bold', 'axes.labelsize': 9, 'axes.labelcolor': PAL['muted'],
    'xtick.labelsize': 8, 'ytick.labelsize': 8, 'xtick.color': PAL['muted'], 'ytick.color': PAL['muted'],
    'legend.fontsize': 8, 'legend.frameon': False, 'lines.linewidth': 2})
# версии библиотек и используемый парсер JSON
print('pandas', pd.__version__, '| numpy', np.__version__, '| парсер JSON:', _json_loads.__module__)

## 1. Вспомогательные функции
Нормализация ключей (ИИН с ведущими нулями, коды договоров), справочник правил ПКБ (Первое кредитное бюро), таймеры.

In [ ]:
class Timer:
    # Таймер шага: печатает название шага при входе и время выполнения при выходе
    def __init__(self, msg):
        self.msg = msg
    def __enter__(self):
        self.t0 = time.time()
        print(f'> {self.msg} ...', flush=True)
        return self
    def __exit__(self, *exc):
        print(f'  готово за {time.time() - self.t0:,.1f} с', flush=True)


# текстовые значения, которые считаются пропусками (пустыми ключами)
_NULLS = ['', 'None', 'none', 'nan', 'NaN', 'NAN', 'NONE', 'NULL', 'null', '<NA>', 'NaT']


def norm_iin(s):
    """ИИН/БИН в виде строки из 12 символов; восстанавливает ведущие нули, потерянные при преобразовании в int/float."""
    s = pd.Series(s, copy=False).astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
    short_digits = s.str.fullmatch(r'\d{1,11}').fillna(False).astype(bool)
    s = s.where(~short_digits, s.str.zfill(12))
    return s.where(~s.isin(_NULLS), np.nan).astype(object)


def norm_code(s):
    """Код договора как строка в верхнем регистре без пробелов по краям (одно правило для наших договоров и для contract_code из ПКБ)."""
    s = pd.Series(s, copy=False).astype(str).str.strip().str.upper()
    return s.where(~s.isin([x.upper() for x in _NULLS]), np.nan).astype(object)


_PD2 = int(pd.__version__.split('.')[0]) >= 2   # True, если версия pandas 2 или выше


def to_dt(s):
    """Надёжный разбор дат. pandas >= 2 определяет ОДИН формат по первому значению и превращает значения в другом формате в NaT,
    поэтому ISO8601 запрашивается явно (охватывает 'YYYY-MM-DD', дату со временем, с долями секунды, объекты date/datetime)."""
    s = pd.Series(s, copy=False) if not isinstance(s, pd.Series) else s
    if pd.api.types.is_datetime64_any_dtype(s):
        return s
    if not _PD2:
        return pd.to_datetime(s, errors='coerce')
    out = pd.to_datetime(s, errors='coerce', format='ISO8601')
    lost = out.isna() & s.notna()
    if lost.mean() > 0.01:                       # формат не ISO: разбираем каждое значение отдельно
        out = pd.to_datetime(s, errors='coerce', format='mixed')
    return out


def days(n):
    # n дней в виде pd.Timedelta
    return pd.Timedelta(days=int(n))


# Правила мониторинга ПКБ: (фрагмент rule_name в нижнем регистре, код, семейство, описание).
# Порядок важен: срабатывает первый фрагмент, найденный в названии правила.
# Фрагменты на русском должны совпадать с реальными названиями правил ПКБ - их не менять.
# Семейства: credit_hunger - кредитный голод (заявки, запросы, новые займы), relief - облегчение условий
# (пролонгация, реструктуризация), delinquency - просрочка, positive - позитивные события, status - особые статусы
# клиента, technical - технические, debt_burden - долговая нагрузка, bank_rule - правила относительно нашего банка,
# other - прочие.
RULES = [
    ('кредитной заявки', 'app_online', 'credit_hunger', 'Онлайн-заявка на кредит'),
    ('пролонгация', 'prolong', 'relief', 'Пролонгация займа'),
    ('любой поставщик', 'lookup_other', 'credit_hunger', 'Другой поставщик (не ПКБ) запросил данные по клиенту'),
    ('сумма просрочки', 'ovd_sum_chg', 'delinquency', 'Общая сумма просрочки выросла/снизилась'),
    ('с целью рефинанс', 'new_loan_refin', 'credit_hunger', 'Новый заём с целью рефинансирования'),
    ('появление нового займа', 'new_loan', 'credit_hunger', 'Появился новый заём'),
    ('урегулирование', 'settlement', 'positive', 'Урегулирование задолженности'),
    ('лудоман', 'st_gambler', 'status', 'Статус «Лудоман» (самозапрет на азартные игры)'),
    ('отписка', 'unsubscribe', 'technical', 'Отписка: нет действующих договоров (техническое событие)'),
    ('призывник', 'st_recruit', 'status', 'Статус «Призывник» (призыв на воинскую службу)'),
    ('залогу', 'collateral_chg', 'other', 'Изменение по залогу'),
    ('запросов на кредитный отчет', 'report_req', 'credit_hunger', 'Запрос кредитного отчёта'),
    ('новый контракт', 'new_contract', 'credit_hunger', 'Появился новый договор'),
    ('реструктур', 'restruct', 'relief', 'Заём реструктурирован / рефинансирован / пересмотрен'),
    ('стоп-кредит', 'st_stopcredit', 'status', 'Статус «Стоп-Кредит» (добровольный запрет на получение займов)'),
    ('статуса по кредиту изменился', 'debt_terminated', 'delinquency', 'Обязательство прекращено без полного погашения'),
    ('частичная просрочка', 'partial_ovd', 'bank_rule', 'Просрочка у нас при платежах другим кредиторам'),
    ('банкрот', 'st_bankrupt', 'status', 'Статус «Банкротство»'),
    ('переуступка', 'loan_assigned', 'delinquency', 'Заём закрыт переуступкой (продан)'),
    ('закрытие займа', 'loan_closed', 'positive', 'Заём закрыт (погашен)'),
    ('сделать сравнение', 'refin_out_80', 'bank_rule', 'Погашено >=80% основного долга у нас после нового займа у другого кредитора'),
    ('ежемесячного платежа', 'instalment_chg', 'debt_burden', 'Изменился общий ежемесячный платёж'),
    ('дней просрочки', 'dpd_chg', 'delinquency', 'Изменилось количество дней просрочки'),
    ('основного долга', 'debt_chg', 'debt_burden', 'Общий основной долг вырос/снизился'),
]
FAMILY = {code: fam for _, code, fam, _ in RULES}          # код правила -> семейство
RULE_DESC = {code: desc for _, code, _, desc in RULES}     # код правила -> описание
STATUS_RULES = [code for _, code, fam, _ in RULES if fam == 'status']   # коды статусных правил


def map_rule_names(names):
    """{rule_name: код}. Неизвестные названия правил получают автоматический код, чтобы по ним всё равно строились признаки-счётчики."""
    out = {}
    for nm in names:
        low = str(nm).lower()
        for frag, code, _, _ in RULES:
            if frag in low:
                out[nm] = code
                break
        else:
            out[nm] = 'other_' + re.sub(r'\W+', '_', low).strip('_')[:30]
            FAMILY.setdefault(out[nm], 'unknown')
            RULE_DESC.setdefault(out[nm], str(nm)[:80])
    return out


def iv_strength(iv):
    # Сила признака по IV: < 0.02 бесполезный, 0.02-0.1 слабый, 0.1-0.3 средний, 0.3-0.5 сильный,
    # >= 0.5 подозрительно высокая IV: проверьте утечку данных (leakage).
    # Возвращаемые значения на английском - это коды категорий столбца strength в таблицах.
    if not np.isfinite(iv):
        return ''
    return ('not useful' if iv < 0.02 else 'weak' if iv < 0.1 else 'medium' if iv < 0.3
            else 'strong' if iv < 0.5 else 'suspicious: check leakage')

# Вспомогательные функции для JSON (используются в предварительных проверках и в парсере событий в разделе 7)
NUM_KEYS = ('overdue_sum', 'debt_sum', 'instalment', 'overdue_instalment_count')   # числовые поля JSON в порядке приоритета
NAN = float('nan')
_EMPTY_NEW = (None, NAN, NAN, NAN, NAN, NAN, NAN, NAN)   # результат разбора пустого/некорректного new_value
_EMPTY_OLD = (NAN, NAN)                                  # результат разбора пустого/некорректного old_value


def _num(v):
    # значение из JSON -> число: bool и 'true'/'false' -> 1/0, строки с пробелами и десятичной запятой -> float, иначе NaN
    if v is None:
        return NAN
    if isinstance(v, bool):
        return 1.0 if v else 0.0
    if isinstance(v, (int, float)):
        return float(v)
    if isinstance(v, str):
        t = v.strip().lower()
        if t in ('true', 'false'):
            return 1.0 if t == 'true' else 0.0
        try:
            return float(t.replace(' ', '').replace(',', '.'))
        except ValueError:
            return NAN
    return NAN


def _id(v):
    # числовой идентификатор: из словаря берётся поле 'id', иначе само значение
    return _num(v.get('id')) if isinstance(v, dict) else _num(v)


def _loads(s):
    # текст JSON -> словарь; None, если текст пустой, некорректный или это не объект JSON
    if isinstance(s, dict):
        return s
    if not isinstance(s, (str, bytes)) or len(s) < 2:
        return None
    try:
        d = _json_loads(s)
    except Exception:
        return None
    return d if isinstance(d, dict) else None


def _flag(d):
    # первый флаг вида is_* / has_* (или stopcredit) как 1/0; NaN, если флага нет
    for k, v in d.items():
        if k.startswith(('is_', 'has_')) or k == 'stopcredit':
            x = _num(v)
            if x == x:                           # x == x ложно только для NaN
                return x
    return NAN


def _first_num(d):
    # первое заполненное числовое поле из NUM_KEYS
    for k in NUM_KEYS:
        v = d.get(k)
        if v is not None:
            return _num(v)
    return NAN


def _parse_new(s):
    # разбор new_value: (код договора, тип кредитора, первое числовое поле, флаг, сумма займа,
    #                    задолженность, остаток долга, длительность пролонгации в днях)
    d = _loads(s)
    if d is None:
        return _EMPTY_NEW
    code = d.get('contract_code')
    code = str(code).strip().upper() if code not in (None, '') else None
    pdays = NAN
    a, b = d.get('prolongation_start_date'), d.get('prolongation_end_date')
    if a and b:
        try:
            pdays = float((date.fromisoformat(str(b)[:10]) - date.fromisoformat(str(a)[:10])).days)
        except ValueError:
            pass
    return (code, _id(d.get('creditor_type')), _first_num(d), _flag(d), _num(d.get('summa_zaima')),
            _num(d.get('zadolzhenost')), _num(d.get('outstanding_debt')), pdays)


def _parse_old(s):
    # разбор old_value: (первое числовое поле, флаг)
    d = _loads(s)
    if d is None:
        return _EMPTY_OLD
    return (_first_num(d), _flag(d))

## 2. Данные
`DEMO_MODE = True` генерирует синтетический портфель с теми же столбцами, типами данных (dtypes) и структурой JSON, что и
`ovrd_contr`, `hes` и `contracts` (около 60 тыс. клиентов, 9 недель мониторинга). Генератор закладывает в данные заранее
известный сигнал, чтобы можно было проверить каждый шаг анализа.

При `DEMO_MODE = False` ноутбук использует объекты, уже загруженные в память. Альтернативные способы загрузки показаны в комментариях в ячейке.

In [ ]:
CT = {1: {'id': '1', 'name': 'Банк второго уровня'}, 2: {'id': '2', 'name': 'Микрофинансовая организация'}}
SYN_RULE_NAMES = {   # тексты rule_name, которые использует генератор (реальные названия правил сопоставляются по фрагментам, см. RULES)
    'app_online': 'Мониторинг кредитной заявки (онлайн заявки)',
    'prolong': 'Пролонгация по займу',
    'lookup_other': 'Любой поставщик (кроме ПКБ) искал информацию по данному субъекту',
    'ovd_sum_chg': 'Совокупная сумма просрочки увеличилась/уменьшилась',
    'new_loan': 'Мониторинг появление нового займа',
    'settlement': 'Мониторинг факта урегулирование задолженности',
    'st_gambler': 'Мониторинг статуса «Лудоман»',
    'unsubscribe': 'Отписка субъектов без активных контрактов',
    'st_recruit': 'Мониторинг статуса «Призывник»',
    'collateral_chg': 'Изменения по залогу займа',
    'report_req': 'Мониторинг запросов на кредитный отчет',
    'new_contract': 'У субъекта появился новый контракт',
    'restruct': 'Мониторинг изменения по кредиту - (Реструктурирован/Рефинансирован/Пересмотрен)',
    'new_loan_refin': 'Появление нового займа с целью Рефинансирование, где получатель единственный кредитор',
    'st_stopcredit': 'Мониторинг статуса «Стоп-Кредит»',
    'debt_terminated': 'Мониторинг статуса по кредиту изменился – Ссудная задолженность прекращена без полного исполнения обязательства',
    'partial_ovd': 'Частичная просрочка по кредиторам (заемщик допустил просрочку в организации получателя, но продолжает платить в других)',
    'st_bankrupt': 'Мониторинг статуса «Банкротства»',
    'loan_closed': 'Мониторинг закрытие займа (погашение)',
    'loan_assigned': 'Мониторинг закрытие займа – переуступка',
    'refin_out_80': 'Сделать сравнение, у кого до погашения не менее 80% от ОД займа – появился кредит в другом банке',
    'instalment_chg': 'Совокупная сумма ежемесячного платежа изменилась',
    'dpd_chg': 'Мониторинг изменения дней просрочки займа',
    'debt_chg': 'Совокупная сумма основного долга увеличилась/уменьшилась',
}


def _j(d):
    return json.dumps(d, ensure_ascii=False)


def _ext_codes(r, n):
    """Коды договоров других кредиторов в нескольких форматах."""
    k = r.integers(0, 5, n)
    x = r.integers(10**6, 10**9, n)
    fmt = ['KZ{:010d}', 'CO2026{:012d}', 'Z{:08d}A0219-3', 'КАРТ-{:010d}', '{:d}']
    return [fmt[a].format(b) for a, b in zip(k, x)]


def _payloads(code, sv, r, codes=None, dpd_old=None, dpd_new=None):
    """JSON-строки old_value / new_value для n событий одного правила (структура JSON скопирована из реальной таблицы)."""
    n = len(sv)
    mfo = r.random(n) < 1 / (1 + np.exp(-(-0.9 + 0.9 * sv)))
    ct = [CT[2] if m else CT[1] for m in mfo]
    cc = codes if codes is not None else _ext_codes(r, n)
    amt = np.round(np.exp(r.normal(12.2 + 0.25 * sv, 0.8)), -3)
    none = [None] * n
    if code == 'app_online':
        new = [_j({'object': {'id': None, 'name': None}, 'purpose': {'id': '04', 'name': 'Потребительские цели'},
                   'summa_zaima': str(int(a)), 'credit_app_id': int(i), 'creditor_type': c})
               for a, i, c in zip(amt, r.integers(10**8, 10**9, n), ct)]
        return none, new
    if code in ('prolong', 'new_loan_refin', 'loan_closed'):
        debt = np.round(amt * r.uniform(0.2, 0.95, n), 0)
        new = []
        for a, d_, c, k, p in zip(amt, debt, ct, cc, r.integers(0, 10, n)):
            d = {'object': {'id': '10', 'name': 'Прочие'},
                 'purpose': {'id': '16', 'name': 'Рефинансирование'} if code == 'new_loan_refin' else {'id': '09', 'name': 'Прочие'},
                 'summa_zaima': str(int(a)) if code != 'loan_closed' else float(a), 'contract_code': k, 'creditor_type': c}
            if code != 'loan_closed':
                d['zadolzhenost'] = str(int(d_))
            if code == 'prolong':
                d['prolongation_start_date'] = '2026-09-08T00:00:00+05:00'
                d['prolongation_end_date'] = f'2026-09-{14 + int(p):02d}T00:00:00+05:00'
            if code == 'new_loan_refin':
                d.update({'platezh': None, 'valuta_platezha': None})
            new.append(_j(d))
        return none, new
    if code == 'lookup_other':
        return none, [_j({'report': {'id': '6', 'name': 'Кредитный отчет – расширенный физическое лицо'}})] * n
    if code == 'report_req':
        return none, [_j({'creditor_type': c, 'credit_report_id': str(i)}) for c, i in zip(ct, r.integers(10**8, 10**9, n))]
    if code == 'ovd_sum_chg':
        o = np.round(np.exp(r.normal(10 + 0.4 * sv, 1.0)), 2) * (r.random(n) < 0.7)
        up = r.random(n) < 1 / (1 + np.exp(-0.9 * sv))
        nv = np.where(up, o + np.round(np.exp(r.normal(9.5, 0.8, n)), 2), o * r.uniform(0, 0.6, n))
        old = [_j({'overdue_sum': float(a), 'contract_code': k, 'creditor_type': c}) for a, k, c in zip(o, cc, ct)]
        new = [_j({'overdue_sum': float(round(b, 2)), 'contract_code': k, 'creditor_type': c}) for b, k, c in zip(nv, cc, ct)]
        return old, new
    if code in ('new_loan', 'restruct', 'collateral_chg', 'debt_terminated', 'loan_assigned'):
        return none, [_j({'contract_code': k, 'creditor_type': c, 'overdue_instalment_count': None}) for k, c in zip(cc, ct)]
    if code == 'new_contract':
        old = [_j({'end_date': None, 'start_date': None, 'subject_role': {'id': None, 'name': None},
                   'contract_code': None, 'financial_institution': {'id': 0, 'name': None}})] * n
        new = [_j({'end_date': '2028-09-01', 'start_date': '2026-09-01', 'subject_role': {'id': 1, 'name': 'Заемщик'},
                   'contract_code': k, 'financial_institution': {'id': int(f), 'name': 'ФИ'}})
               for k, f in zip(cc, r.integers(1, 400, n))]
        return old, new
    if code == 'debt_chg':
        o = np.round(np.exp(r.normal(13, 1.0, n)), 2)
        up = r.random(n) < 1 / (1 + np.exp(-(-1.6 + 0.6 * sv)))
        nv = np.where(up, o * r.uniform(1.0, 1.3, n), o * r.uniform(0.9, 1.0, n))
        old = [_j({'debt_sum': float(a), 'contract_code': k, 'creditor_type': c}) for a, k, c in zip(o, cc, ct)]
        new = [_j({'debt_sum': float(round(b, 2)), 'contract_code': k, 'creditor_type': c}) for b, k, c in zip(nv, cc, ct)]
        return old, new
    if code == 'instalment_chg':
        o = np.round(np.exp(r.normal(11, 0.8, n)), 2)
        nv = o * np.where(r.random(n) < 1 / (1 + np.exp(-0.5 * sv)), r.uniform(1.0, 1.5, n), r.uniform(0.6, 1.0, n))
        old = [_j({'instalment': str(a)}) for a in o]
        new = [_j({'instalment': str(round(b, 2)), 'contract_code': k, 'creditor_type': c, 'outstanding_debt': str(round(b * 12, 2))})
               for b, k, c in zip(nv, cc, ct)]
        return old, new
    if code == 'dpd_chg':
        if dpd_old is None:
            dpd_old = np.floor(r.exponential(8 * np.exp(0.5 * sv))).astype(int)
            worse = r.random(n) < 1 / (1 + np.exp(-(0.2 + 0.9 * sv)))
            dpd_new = np.where(worse, dpd_old + r.integers(1, 31, n), 0)
        old = [_j({'contract_code': k, 'creditor_type': c, 'overdue_instalment_count': int(a)}) for k, c, a in zip(cc, ct, dpd_old)]
        new = [_j({'contract_code': k, 'creditor_type': c, 'overdue_instalment_count': int(b)}) for k, c, b in zip(cc, ct, dpd_new)]
        return old, new
    flags = {'st_gambler': 'is_gambler', 'st_recruit': 'is_recruit', 'st_stopcredit': 'stopcredit', 'st_bankrupt': 'is_bankrupt'}
    if code in flags:
        return [_j({flags[code]: 0})] * n, [_j({flags[code]: 1})] * n
    if code == 'settlement':
        return none, [_j({'is_settlement': 1})] * n
    if code == 'partial_ovd':
        return none, [_j({'has_half_overdue_bank_and_not_overdue_non_bank': True})] * n
    if code == 'refin_out_80':
        return none, [_j({'has_refinance_more_80_percent_ood': True})] * n
    if code == 'unsubscribe':
        return ([_j({'reason': None, 'status': 'Active', 'group_id': 66, 'removed_at': None})] * n,
                [_j({'reason': 'NO_ACTIVE_CONTRACTS', 'status': 'Removed', 'group_id': 66,
                     'removed_at': '2026-10-19T23:08:40.846181+05:00'})] * n)
    return none, none


def make_synthetic_data(n_clients=60_000, seed=SEED):
    """Синтетические ovrd_contr / hes / contracts со схемой, как в промышленных данных, и заранее заложенным сигналом."""
    r = np.random.default_rng(seed)
    D = np.timedelta64(1, 'D')
    issue_from, issue_to = np.datetime64('2025-01-01'), np.datetime64('2026-10-15')
    data_end = np.datetime64('2026-11-18')
    hes_from, hes_to = np.datetime64('2026-09-07'), np.datetime64('2026-11-18')   # события создаются по 2026-11-17 включительно

    # клиенты: z — внутренний риск, s — финансовый стресс, видимый у других кредиторов (коррелирует с z)
    z = r.normal(size=n_clients)
    s = 0.5 * z + np.sqrt(0.75) * r.normal(size=n_clients)
    iin = np.array([f'{a:02d}{b:010d}' for a, b in zip(r.integers(60, 106, n_clients) % 100,
                                                       r.integers(0, 10**10, n_clients))])
    # договоры
    n_c = 1 + r.poisson(0.45, n_clients)
    ci = np.repeat(np.arange(n_clients), n_c)
    N = len(ci)
    open_d = issue_from + r.integers(0, int((issue_to - issue_from) / D) + 1, N) * D
    prod = r.choice(np.array(['16007', '16062', '16167', '16168']), N, p=[.35, .15, .25, .25])
    term_m = np.where(prod == '16167', r.choice([6, 12, 18], N),
             np.where(prod == '16168', r.choice([12, 24], N),
             np.where(prod == '16062', r.choice([12, 18, 24, 36], N), r.choice([24, 36, 48, 60], N))))
    end_d = open_d + np.round(term_m * 30.44).astype(int) * D
    mu = np.where(prod == '16167', 10.5, np.where(prod == '16168', 13.0, np.where(prod == '16062', 13.5, 14.5)))
    amount = np.round(np.exp(r.normal(mu, 0.6)), -2)
    early = r.random(N) < 0.2
    life = ((end_d - open_d) / D).astype(int)
    close_d = end_d.copy()
    close_d[early] = open_d[early] + np.maximum(30, (life[early] * r.uniform(0.15, 0.9, early.sum())).astype(int)) * D
    status = np.where(close_d <= data_end, 'Погашен', 'Актуален')
    pref = r.choice(np.array(['3199191', '1199221', '1340260', '3277770', '1677770', '2477770']), N)
    contract_number = np.array([f'{p}{i:09d}' for p, i in zip(pref, r.permutation(N) + 100_000)])
    pr = np.where(np.isin(prod, ['16007', '16062']), 'EKP', np.where(prod == '16167', 'CCC', 'CPA'))
    process_ref = np.array([f'{p}{i:09d}' if p == 'EKP' else f'{p}{i:012d}'
                            for p, i in zip(pr, r.permutation(N) + 10_000_000)])
    merchants = np.array([f'{x:012d}' for x in r.integers(10**11, 10**12, 300)])
    partner = np.where(prod == '16167', 'marketplace', np.where(prod == '16168', r.choice(merchants, N), None)).astype(object)
    prod_eff = pd.Series(prod).map({'16007': 0.0, '16062': 0.2, '16167': 0.35, '16168': 0.15}).values

    # эпизоды просрочки
    first_due = open_d + 30 * D
    active_days = np.maximum(0, ((np.minimum(close_d, data_end) - first_due) / D).astype(int))
    n_ep = r.poisson(0.0030 * np.exp(0.8 * z[ci] + prod_eff) * active_days)
    e_c = np.repeat(np.arange(N), n_ep)
    e_start = first_due[e_c] + (r.random(len(e_c)) * active_days[e_c]).astype(int) * D
    zc, sc = z[ci[e_c]], s[ci[e_c]]
    roll = r.random(len(e_c)) < 1 / (1 + np.exp(-(-2.4 + 1.0 * zc + 0.6 * sc + prod_eff[e_c])))
    e_dpd = np.where(roll, 31 + r.exponential(45, len(e_c)).astype(int),
                     1 + np.floor(29.999 * r.beta(0.7, np.clip(2.6 - 0.5 * sc, 1.2, 4.0))).astype(int))
    E = pd.DataFrame({'c': e_c, 'start': e_start, 'dpd': e_dpd, 'roll': roll}).sort_values(['c', 'start'])
    E['end'] = E['start'] + pd.to_timedelta(E['dpd'] - 1, unit='D')
    E['cm'] = E.groupby('c')['end'].cummax()
    prev = E.groupby('c')['cm'].shift()
    E = E[prev.isna() | (E['start'] > prev + pd.Timedelta(days=1))]
    E = E[E['start'] <= pd.Timestamp(data_end)].reset_index(drop=True)
    E['dpd_obs'] = np.minimum(E['dpd'], (pd.Timestamp(data_end) - E['start']).dt.days + 1)

    # строки дней просрочки (DPD): последний день эпизода, концы месяцев и ~35% остальных дней (проверка разреженной выгрузки)
    L = E['dpd_obs'].values.astype(int)
    rep = np.repeat(np.arange(len(E)), L)
    off = np.arange(L.sum()) - np.repeat(np.cumsum(L) - L, L)
    act = E['start'].values.astype('datetime64[D]')[rep] + off * D
    is_last = off == (L[rep] - 1)
    is_month_end = (act + D).astype('datetime64[M]') != act.astype('datetime64[M]')
    keep = is_last | is_month_end | (r.random(len(off)) < 0.35)
    cidx = E['c'].values[rep][keep]
    act_k = act[keep]
    ns = lambda a: a.astype('datetime64[ns]')
    ovrd = pd.DataFrame({
        'rep_month': np.char.add(act_k.astype('datetime64[M]').astype(str), '-01').astype(object),
        'act_date': act_k.astype(str).astype(object),
        'contract_number': contract_number[cidx].astype(object),
        'ovd_days_cnt': (off[keep] + 1).astype(float),
        'process_ref': process_ref[cidx].astype(object),
        'iin': iin[ci[cidx]].astype(object),
        'open_date': ns(open_d[cidx]), 'close_date': ns(close_d[cidx]), 'end_date': ns(end_d[cidx]),
        'status_name': status[cidx].astype(object),
        'contract_amount': amount[cidx].astype(float),
        'partner_name': partner[cidx],
        'product_id': prod[cidx].astype(object)})
    dup = ovrd.sample(frac=0.01, random_state=seed).assign(status_name='Погашен')   # дубликаты от слияния, как в реальных данных
    ovrd = pd.concat([ovrd, dup], ignore_index=True).sort_values(['rep_month', 'act_date'], kind='mergesort').reset_index(drop=True)

    contracts = pd.DataFrame({
        'contract_number': contract_number.astype(object), 'process_ref': process_ref.astype(object),
        'iin': iin[ci].astype(object), 'open_date': ns(open_d), 'close_date': ns(close_d), 'end_date': ns(end_d),
        'status_name': status.astype(object), 'contract_amount': amount, 'partner_name': partner,
        'product_id': prod.astype(object)})

    # события мониторинга ПКБ
    win_days = float((hes_to - hes_from) / D)
    act_client = np.zeros(n_clients, bool)
    act_client[ci[(open_d < hes_to) & (close_d > hes_from)]] = True
    rates = {   # события по другим кредиторам: (дневная интенсивность, влияние стресса s)
        'app_online': (0.010, 0.7), 'lookup_other': (0.004, 0.5), 'report_req': (0.007, 0.6),
        'new_loan': (0.003, 0.5), 'new_contract': (0.003, 0.4), 'new_loan_refin': (0.0008, 0.6),
        'ovd_sum_chg': (0.004, 1.0), 'dpd_chg': (0.006, 1.1), 'debt_chg': (0.030, 0.1),
        'instalment_chg': (0.010, 0.2), 'collateral_chg': (0.001, 0.0), 'loan_closed': (0.005, -0.3),
        'settlement': (0.0008, 0.3), 'loan_assigned': (0.0003, 1.3), 'debt_terminated': (0.0002, 1.3),
        'prolong': (0.002, 0.8), 'restruct': (0.0005, 0.8), 'st_gambler': (0.0002, 0.9),
        'st_recruit': (0.0002, 0.0), 'st_stopcredit': (0.0002, 0.5), 'st_bankrupt': (0.00005, 1.5),
        'refin_out_80': (0.0004, -0.5), 'unsubscribe': (0.0003, 0.0)}
    parts = []

    def add(code, cl, t, codes=None, dpd_old=None, dpd_new=None):
        if len(cl) == 0:
            return
        old, new = _payloads(code, s[cl], r, codes, dpd_old, dpd_new)
        parts.append(pd.DataFrame({'cl': cl, 'code': code, 't': t, 'old': old, 'new': new}))

    t_unif = lambda n: pd.Timestamp(hes_from) + pd.to_timedelta(r.random(n) * win_days * 86400, unit='s')
    for code, (rate, beta) in rates.items():
        lam = rate * np.exp(beta * s - beta ** 2 / 2) * win_days * np.where(act_client, 1.0, 0.15)
        cl = np.repeat(np.arange(n_clients), r.poisson(lam))
        add(code, cl, t_unif(len(cl)))

    # события по нашим договорам в период просрочки и всплески стресса перед началом эпизода просрочки
    E['cl'] = ci[E['c'].values]
    hes_from_ts, hes_to_ts = pd.Timestamp(hes_from), pd.Timestamp(hes_to)
    Ew = E[(E['start'] > hes_from_ts - pd.Timedelta(days=60)) & (E['start'] < hes_to_ts)]
    own_cl, own_t, own_code, d_old, d_new = [], [], [], [], []
    for row in Ew.itertuples(index=False):
        k = contract_number[row.c] if r.random() < 0.6 else process_ref[row.c]
        for step in range(1, int(row.dpd_obs), 7):
            t = row.start + pd.Timedelta(days=step + 1, hours=float(r.uniform(0, 20)))
            if hes_from_ts <= t < hes_to_ts:
                own_cl.append(row.cl); own_t.append(t); own_code.append(k)
                d_old.append(max(step - 7, 0)); d_new.append(step)
    add('dpd_chg', np.array(own_cl, int), pd.to_datetime(own_t), own_code, np.array(d_old), np.array(d_new))
    m = (Ew['start'] + pd.Timedelta(days=2) >= hes_from_ts) & (Ew['start'] + pd.Timedelta(days=2) < hes_to_ts)
    po = Ew[m & (r.random(len(Ew)) < 0.35)]
    add('partial_ovd', po['cl'].values, po['start'] + pd.Timedelta(days=2, hours=9))
    Eb = E[(E['start'] >= hes_from_ts + pd.Timedelta(days=5)) & (E['start'] < hes_to_ts)]
    nb = r.poisson(np.where(Eb['roll'], 0.7, 0.35))
    bcl = np.repeat(Eb['cl'].values, nb)
    bst = np.repeat(Eb['start'].values, nb)
    bcode = r.choice(['app_online', 'report_req', 'dpd_chg', 'ovd_sum_chg', 'new_loan', 'lookup_other'],
                     len(bcl), p=[.3, .25, .15, .1, .1, .1])
    bt = pd.to_datetime(bst) - pd.to_timedelta(r.uniform(1, 21, len(bcl)) * 86400, unit='s')
    for code in np.unique(bcode):
        mm = bcode == code
        add(code, bcl[mm], bt[mm])

    ev = pd.concat(parts, ignore_index=True)
    ev = ev[(ev['t'] >= hes_from_ts) & (ev['t'] < hes_to_ts)].reset_index(drop=True)
    n = len(ev)
    created = pd.to_datetime(ev['t'].values)
    dv = (created.normalize() + pd.to_timedelta(r.integers(0, 2, n), unit='D'))
    hes = pd.DataFrame({
        'date_value': dv.strftime('%Y-%m-%d').astype(object),
        'date_change': dv + pd.Timedelta(days=1, hours=14, minutes=13, seconds=47),
        'internal_event_id': [str(uuid.UUID(bytes=r.bytes(16))) for _ in range(n)],
        'bin_iin': iin[ev['cl'].values].astype(object),
        'rule_name': ev['code'].map(SYN_RULE_NAMES).values.astype(object),
        'old_value': ev['old'].values, 'new_value': ev['new'].values,
        'event_date': created - pd.to_timedelta(r.uniform(0, 1.5, n) * 86400, unit='s'),
        'created_date': created})
    hes = hes.sort_values(['date_value', 'internal_event_id']).reset_index(drop=True)
    return ovrd, hes, contracts


if DEMO_MODE:
    with Timer('Генерация синтетических демо-данных'):
        ovrd_contr, hes, contracts = make_synthetic_data()
else:
    # Ваши датафреймы берутся из этого ядра по имени (ничего не копируется).
    # Если они загружены в другом ноутбуке, вставьте эти ячейки туда либо сохраните их и прочитайте здесь, например:
    #   ovrd_contr = pd.read_pickle('ovrd_contr.pkl'); hes = pd.read_pickle('hes.pkl')
    # hes может быть и итератором по частям (чанкам), например pd.read_sql(sql, conn, chunksize=2_000_000);
    # тогда он фильтруется по частям в разделе 7 и никогда не хранится в памяти целиком (HES_START задайте вручную).
    _g = globals()
    _missing = [v for v in (OVRD_VAR, HES_VAR) if v not in _g]
    if _missing:
        raise NameError(f'Датафреймов {_missing} нет в этом ядре. Сначала загрузите их (или измените OVRD_VAR / HES_VAR), '
                        'либо задайте DEMO_MODE = True, чтобы запустить демо на синтетических данных.')
    ovrd_contr = _g[OVRD_VAR]
    hes = _g[HES_VAR]
    contracts = _g.get(CONTRACTS_VAR)
    if not isinstance(contracts, pd.DataFrame):
        contracts = None

print('ovrd_contr:', ovrd_contr.shape, '| contracts:', None if contracts is None else contracts.shape,
      '| hes:', hes.shape if isinstance(hes, pd.DataFrame) else 'итератор по частям')
if DEMO_MODE:
    display(ovrd_contr.head(3))
    display(hes.head(3))

## 2b. Предварительная проверка (preflight check) входных датафреймов
Быстрые проверки на выборках строк до начала тяжёлых вычислений: наличие обязательных колонок, распознавание дат и дней
просрочки (DPD), формат ИИН (IIN), пересечение клиентов между `hes` и `ovrd_contr`, доля успешно разобранных JSON по каждому
правилу, распознанные названия правил, а также совпадают ли коды договоров в ПКБ (Первое кредитное бюро, PKB) с номерами
наших договоров. Статус «ОШИБКА» (`FAIL`) останавливает ноутбук (`STOP_ON_PREFLIGHT_FAIL`), «ПРЕДУПРЕЖДЕНИЕ» (`WARN`) носит только
информационный характер.

In [ ]:
def _sample_rows(df, n, seed=SEED):
    if len(df) <= n:
        return df
    pos = np.unique(np.random.default_rng(seed).integers(0, len(df), size=n))   # быстро даже на 85 млн строк
    return df.iloc[pos]


def preflight(ovrd, hes, contracts):
    rows = []

    def add(obj, check, status, detail):
        rows.append({'object': obj, 'check': check, 'status': status, 'detail': detail})

    def cols(df, obj, need, opt):
        miss = [c for c in need if c not in df.columns]
        add(obj, 'обязательные колонки', 'FAIL' if miss else 'OK', f'отсутствуют: {miss}' if miss else ', '.join(need))
        mo = [c for c in opt if c not in df.columns]
        if mo:
            add(obj, 'необязательные колонки', 'WARN', f'отсутствуют: {mo} (признаки на их основе будут пустыми)')
        return not miss

    if not isinstance(ovrd, pd.DataFrame):
        add('ovrd_contr', 'тип', 'FAIL', f'ожидался DataFrame, получен {type(ovrd).__name__}')
    elif cols(ovrd, 'ovrd_contr', [C_CONTRACT, C_IIN, C_ACT_DATE, C_DPD],
              [C_PROCESS_REF, C_OPEN, C_CLOSE, C_END, C_STATUS, C_AMOUNT, C_PARTNER, C_PRODUCT]):
        s = _sample_rows(ovrd, DQ_SAMPLE)
        ad = to_dt(s[C_ACT_DATE])
        r = ad.notna().mean()
        add('ovrd_contr', f'{C_ACT_DATE} распознаётся как дата', 'OK' if r > 0.99 else 'WARN' if r > 0.9 else 'FAIL',
            f'{r:.2%} распознано, диапазон {ad.min()} .. {ad.max()}')
        d = pd.to_numeric(s[C_DPD], errors='coerce')
        add('ovrd_contr', f'{C_DPD} числовое и > 0', 'OK' if (d > 0).mean() > 0.95 else 'WARN',
            f'{(d > 0).mean():.2%} > 0, {d.isna().mean():.2%} пропусков, максимум {d.max():,.0f}')
        ii = norm_iin(s[C_IIN])
        r = ii.str.fullmatch(r'\d{12}').fillna(False).astype(bool).mean()
        add('ovrd_contr', f'{C_IIN} = 12 цифр', 'OK' if r > 0.99 else 'WARN' if r > 0.8 else 'FAIL', f'{r:.2%}')
        cn = norm_code(s[C_CONTRACT])
        add('ovrd_contr', f'{C_CONTRACT} заполнен', 'OK' if cn.notna().mean() > 0.99 else 'WARN', f'{cn.notna().mean():.2%}')
    if contracts is not None:
        cols(contracts, 'contracts', [C_CONTRACT, C_IIN, C_OPEN], [C_PROCESS_REF, C_CLOSE, C_END, C_STATUS, C_AMOUNT,
                                                                  C_PARTNER, C_PRODUCT])
    else:
        add('contracts', 'наличие', 'WARN', f'`{CONTRACTS_VAR}` не найден: часть B (раннее предупреждение, EWS) пропускается')
    if isinstance(hes, pd.DataFrame):
        if cols(hes, 'hes', [H_IIN, H_RULE, H_NEW, EVENT_TIME_COL], [H_OLD, H_ID, H_EVENT_DATE]):
            s = _sample_rows(hes, min(DQ_SAMPLE, 300_000))
            t = s[EVENT_TIME_COL] if pd.api.types.is_datetime64_any_dtype(s[EVENT_TIME_COL]) else to_dt(s[EVENT_TIME_COL])
            r = t.notna().mean()
            add('hes', f'{EVENT_TIME_COL} распознаётся как дата', 'OK' if r > 0.99 else 'WARN' if r > 0.9 else 'FAIL',
                f'{r:.2%} распознано, диапазон {t.min()} .. {t.max()}')
            hi = norm_iin(s[H_IIN])
            r = hi.str.fullmatch(r'\d{12}').fillna(False).astype(bool).mean()
            add('hes', f'{H_IIN} = 12 цифр', 'OK' if r > 0.99 else 'WARN' if r > 0.8 else 'FAIL', f'{r:.2%}')
            if isinstance(ovrd, pd.DataFrame) and C_IIN in ovrd.columns:
                known = set(norm_iin(pd.Series(pd.unique(ovrd[C_IIN].values))).dropna())
                r = hi.isin(known).mean()
                add('hes', 'клиенты есть также в ovrd_contr', 'OK' if r > 0.05 else 'FAIL',
                    f'{r:.2%} событий выборки относятся к клиентам с просрочкой (0% означает, что ключи ИИН не совпадают)')
            rm = map_rule_names(pd.unique(s[H_RULE].dropna()))
            unk = [k for k, v in rm.items() if v.startswith('other_')]
            add('hes', 'названия правил распознаны', 'OK' if not unk else 'WARN',
                f'{len(rm) - len(unk)} из {len(rm)} распознано' + (f'; неизвестные (только подсчитываются): {unk[:5]}' if unk else ''))
            parsed = s[H_NEW].map(lambda v: _loads(v) is not None)
            by_rule = parsed.groupby(s[H_RULE].map(rm)).mean()
            bad = by_rule[by_rule < 0.5]
            add('hes', 'JSON в new_value разбирается', 'OK' if parsed.mean() > 0.9 and bad.empty else 'WARN',
                f'{parsed.mean():.2%} в целом' + (f'; низкая доля для {bad.round(2).to_dict()}' if len(bad) else ''))
            if isinstance(ovrd, pd.DataFrame) and C_CONTRACT in ovrd.columns:
                dpd_rows = s[s[H_RULE].map(rm) == 'dpd_chg']
                codes = pd.Series([_parse_new(v)[0] for v in dpd_rows[H_NEW].values], dtype=object).dropna()
                own = set(norm_code(pd.Series(pd.unique(ovrd[C_CONTRACT].values))).dropna())
                if C_PROCESS_REF in ovrd.columns:
                    own |= set(norm_code(pd.Series(pd.unique(ovrd[C_PROCESS_REF].values))).dropna())
                r = codes.isin(own).mean() if len(codes) else np.nan
                add('hes', 'события изменения DPD по нашим договорам', 'OK' if r == r and r > 0.01 else 'WARN',
                    f'{r:.2%} событий изменения DPD в выборке совпадают с нашими contract_number/process_ref '
                    '(около 0% означает, что ПКБ использует другой код; тогда разделение на наши договоры и договоры других кредиторов будет неверным)')
        add('hes', 'память (поверхностная оценка)', 'OK', f'{hes.memory_usage(index=False).sum() / 1e9:,.1f} ГБ без учёта строковых данных')
    else:
        add('hes', 'тип', 'WARN', 'итератор по частям: проверки выполняются на лету; задайте HES_START вручную')
    res = pd.DataFrame(rows)
    display(res)
    return res


with Timer('Предварительные проверки'):
    preflight_result = preflight(ovrd_contr, hes, contracts)
if STOP_ON_PREFLIGHT_FAIL and (preflight_result['status'] == 'FAIL').any():
    raise ValueError('Предварительные проверки не пройдены (см. строки со статусом ОШИБКА (FAIL) выше). Исправьте названия колонок в первой ячейке '
                     'или сами данные, либо задайте STOP_ON_PREFLIGHT_FAIL = False, чтобы всё равно продолжить.')

## 3. Атрибуты договоров и множество кодов наших договоров
В данных ПКБ (Первое кредитное бюро, PKB) поле `contract_code` для наших собственных кредитов приходит либо как `contract_number`
(например, `3577770000000001`), либо как `process_ref` (например, `EKP000000001`), поэтому собираются оба кода. Дублирующиеся
строки договора (после объединения таблиц один и тот же договор указан один раз как «Актуален» и один раз как «Погашен») сводятся
к одной строке, при этом предпочтение отдаётся закрытому статусу.

In [ ]:
def channel_of(partner):
    p = pd.Series(partner, copy=False).fillna('').astype(str).str.strip()
    out = np.where(p.isin(_NULLS), 'direct', np.where(p.str.fullmatch(r'\d{10,12}').fillna(False), 'merchant', p))
    return pd.Series(out, index=p.index)


def build_contract_info(ovrd, contracts=None):
    cols = [C_CONTRACT, C_IIN, C_PROCESS_REF, C_OPEN, C_CLOSE, C_END, C_STATUS, C_AMOUNT, C_PARTNER, C_PRODUCT]
    srcs = [x for x in (contracts, ovrd) if x is not None]
    frames = []
    for i, src in enumerate(srcs):
        c = [k for k in cols if k in src.columns]
        f = src[c].drop_duplicates()
        f = f.assign(_src=i)
        frames.append(f)
    ci = pd.concat(frames, ignore_index=True)
    ci['contract'] = norm_code(ci[C_CONTRACT]).values
    ci = ci.dropna(subset=['contract'])
    closed = ci[C_STATUS].astype(str).str.contains('Погаш|Закры|Спис', case=False, regex=True) if C_STATUS in ci else False
    ci['_closed'] = np.asarray(closed, dtype=int)
    ci = ci.sort_values(['contract', '_src', '_closed'], ascending=[True, True, False])
    ci = ci.groupby('contract', sort=False).first().reset_index()   # первое непустое значение по каждой колонке: сначала из contracts, затем из закрытых строк
    out = pd.DataFrame({'contract': ci['contract'].values})
    out['iin'] = norm_iin(ci[C_IIN]).values if C_IIN in ci else np.nan
    out['process_ref'] = norm_code(ci[C_PROCESS_REF]).values if C_PROCESS_REF in ci else np.nan
    for src, dst in ((C_OPEN, 'open_date'), (C_CLOSE, 'close_date'), (C_END, 'end_date')):
        out[dst] = to_dt(ci[src]).values if src in ci else pd.NaT
    out['amount'] = pd.to_numeric(ci[C_AMOUNT], errors='coerce').values if C_AMOUNT in ci else np.nan
    out['product'] = ci[C_PRODUCT].astype(str).values if C_PRODUCT in ci else 'NA'
    out['channel'] = channel_of(ci[C_PARTNER]).values if C_PARTNER in ci else 'NA'
    out['closed'] = ci['_closed'].values
    # True, если договор взят из полной таблицы `contracts`; о договорах, которые есть только в ovrd_contr, известно
    # лишь начиная с их первой просрочки (использовать их раньше означало бы утечку данных из будущего, leakage)
    out['from_contracts'] = (ci['_src'].values == 0) if contracts is not None else np.zeros(len(ci), dtype=bool)
    return out


with Timer('Атрибуты договоров'):
    cinfo = build_contract_info(ovrd_contr, contracts)
    OWN_CODES = set(cinfo['contract'].dropna()) | set(cinfo['process_ref'].dropna())
print(f'{len(cinfo):,} договоров, {len(OWN_CODES):,} кодов наших договоров')
display(cinfo.head(3))

## 4. Эпизоды просрочки и проверки качества данных в `ovrd_contr`
Для каждой строки `dlq_start = act_date − (ovd_days_cnt − 1)`. Строки одного договора, у которых `dlq_start` совпадает
(с допуском `EPISODE_START_TOL_DAYS` дней), образуют один эпизод просрочки; по его максимальному числу дней просрочки (DPD)
видно, перекатился ли он дальше 30 дней просрочки.
Диагностика показывает, насколько плотно идут строки (ежедневно или с пропусками) и растёт ли DPD на один день за каждый
календарный день.

In [ ]:
def build_episodes(ovrd, tol_days=EPISODE_START_TOL_DAYS, data_end=None):
    d = pd.DataFrame({'contract': norm_code(ovrd[C_CONTRACT]).values,
                      'iin': norm_iin(ovrd[C_IIN]).values,
                      'act_date': to_dt(ovrd[C_ACT_DATE]).values,
                      'dpd': pd.to_numeric(ovrd[C_DPD], errors='coerce').values})
    n0 = len(d)
    d = d.dropna(subset=['contract', 'act_date', 'dpd'])
    d = d[d['dpd'] >= 1]
    if data_end is not None:
        d = d[d['act_date'] <= pd.Timestamp(data_end)]
    d['dpd'] = d['dpd'].round().astype('int32')
    # одна строка на договор-день: объединение со статусами договоров может дублировать строки; оставляем максимальный DPD
    d = (d.groupby(['contract', 'act_date'], sort=False)
           .agg(dpd=('dpd', 'max'), iin=('iin', 'first')).reset_index())
    d['dlq_start'] = d['act_date'] - pd.to_timedelta(d['dpd'] - 1, unit='D')
    d = d.sort_values(['contract', 'dlq_start', 'act_date'], kind='mergesort')
    gap = (d['dlq_start'] - d.groupby('contract')['dlq_start'].shift()).dt.days
    d['ep_seq'] = (gap.isna() | (gap > tol_days)).astype('int32').groupby(d['contract']).cumsum()
    ep = (d.groupby(['contract', 'ep_seq'], sort=False)
            .agg(iin=('iin', 'first'), dlq_start=('dlq_start', 'min'), max_dpd=('dpd', 'max'),
                 first_seen=('act_date', 'min'), last_seen=('act_date', 'max'), n_rows=('dpd', 'size'))
            .reset_index())
    ep = ep.sort_values(['contract', 'dlq_start'], kind='mergesort').reset_index(drop=True)
    # снижение DPD после частичной оплаты даёт «новый» dlq_start внутри ещё не закончившегося эпизода: это не новый выход на просрочку
    ep['overlap'] = (ep['dlq_start'] <= ep.groupby('contract')['last_seen'].shift()).values
    ep['ep_uid'] = np.arange(len(ep))
    stats = {'строк на входе': n0, 'договор-дней': len(d), 'эпизодов': len(ep),
             'договоров с просрочкой': ep['contract'].nunique(), 'клиентов с просрочкой': ep['iin'].nunique()}
    return ep, d, stats


with Timer('Эпизоды просрочки'):
    episodes, dpd_rows, ep_stats = build_episodes(ovrd_contr, data_end=DATA_END)
DATA_END_TS = pd.Timestamp(DATA_END) if DATA_END else dpd_rows['act_date'].max().normalize()
print({k: f'{v:,}' for k, v in ep_stats.items()}, '| конец данных по DPD:', DATA_END_TS.date())

span = (episodes['last_seen'].clip(upper=DATA_END_TS) - episodes['dlq_start']).dt.days + 1
dq = pd.Series({
    'Строк на один день эпизода (1.0 = строка за каждый день)': (episodes['n_rows'] / span.clip(lower=1)).median(),
    'Доля эпизодов, представленных только одной строкой': (episodes['n_rows'] == 1).mean(),
    'Доля договор-дней со смещённой датой начала просрочки (dlq_start)': ((dpd_rows['dlq_start'] - dpd_rows.groupby(['contract', 'ep_seq'])['dlq_start'].transform('min')).dt.days > 0).mean(),
    'Доля эпизодов, достигших DPD 31+': (episodes['max_dpd'] >= 31).mean(),
    'Доля эпизодов, начавшихся внутри открытого эпизода (частичная оплата)': episodes['overlap'].mean(),
    'Медиана максимального DPD': episodes['max_dpd'].median()})
display(dq.to_frame('value'))
del dpd_rows
gc.collect()

## 5. Покрытие мониторинга ПКБ и качество данных
Ежедневные объёмы событий мониторинга ПКБ (Первое кредитное бюро, PKB) показывают, когда мониторинг фактически начался
(первые дни часто загружены лишь частично); задержка между `event_date` (в ПКБ) и `created_date` (в банке) показывает,
насколько свежие данные поступают по каждому правилу.

In [ ]:
RULE_MAP = {}
if isinstance(hes, pd.DataFrame):
    with Timer('Ежедневные и еженедельные объёмы по правилам (вся таблица, подсчёт средствами numpy)'):
        t_col = hes[EVENT_TIME_COL]
        if not pd.api.types.is_datetime64_any_dtype(t_col):
            t_col = to_dt(t_col)
        day_num = t_col.values.astype('datetime64[D]').astype('int64')       # дни с 1970-01-01, NaT -> минимальное значение int
        ok = ~np.isnat(t_col.values)
        d0 = int(day_num[ok].min())
        daily_cnt = np.bincount(day_num[ok] - d0)
        daily = pd.Series(daily_cnt, index=pd.to_datetime(np.arange(len(daily_cnt)) + d0, unit='D'))
        daily = daily[daily > 0]
        if RUN_HES_DQ:
            rcode, runiq = pd.factorize(hes[H_RULE].values)                 # -1 = название правила отсутствует
            RULE_MAP = map_rule_names(runiq)
            wk = (day_num - 4) // 7                                         # недели, начинающиеся с понедельника
            w0 = int(wk[ok].min())
            nw = int(wk[ok].max()) - w0 + 1
            keep = ok & (rcode >= 0)
            cnt = np.bincount(rcode[keep] * nw + (wk[keep] - w0), minlength=len(runiq) * nw).reshape(len(runiq), nw)
            week_start = pd.to_datetime((np.arange(nw) + w0) * 7 + 4, unit='D')
            weekly_rule = (pd.DataFrame(cnt, index=[RULE_MAP[r] for r in runiq],
                                        columns=[d.strftime('%d.%m') for d in week_start])
                             .groupby(level=0).sum())
            del rcode, wk, keep
            if H_EVENT_DATE in hes.columns:
                smp = _sample_rows(hes[[H_RULE, H_EVENT_DATE, EVENT_TIME_COL]], DQ_SAMPLE)
                lag_h = (to_dt(smp[EVENT_TIME_COL]) - to_dt(smp[H_EVENT_DATE])) / np.timedelta64(1, 'h')
                lat = (pd.DataFrame({'rule': smp[H_RULE].map(RULE_MAP).values, 'lag_h': lag_h.values})
                         .groupby('rule')['lag_h'].describe(percentiles=[.5, .9, .99])[['count', '50%', '90%', '99%']])
                del smp
        del t_col, day_num, ok
        gc.collect()
    med = daily.median()
    HES_START_TS = pd.Timestamp(HES_START) if HES_START else daily[daily >= 0.5 * med].index.min()
    HES_END_TS = daily.index.max()
    print(f'событий: {len(hes):,} | первый день {daily.index.min().date()} | полное покрытие определено с {HES_START_TS.date()} '
          f'| последний день {HES_END_TS.date()}')
    fig, ax = plt.subplots(figsize=(10, 2.6))
    ax.plot(daily.index, daily.values, color=PAL['blue'])
    ax.axvline(HES_START_TS, color=PAL['gray'], lw=0.8)
    ax.set_title('События мониторинга ПКБ по дням (все правила)')
    ax.set_ylabel('число событий')
    plt.tight_layout(); plt.show()
    if RUN_HES_DQ:
        weekly_rule.insert(0, 'description', [RULE_DESC.get(c, '') for c in weekly_rule.index])
        print('События по правилам и неделям (в заголовке колонки — начало недели, понедельник):')
        display(weekly_rule)
        if H_EVENT_DATE in hes.columns:
            print(f'Задержка {EVENT_TIME_COL} - {H_EVENT_DATE}, в часах (выборка из {DQ_SAMPLE:,} строк):')
            display(lat)
else:
    if HES_START is None:
        raise ValueError('hes является итератором: задайте HES_START явно')
    HES_START_TS = pd.Timestamp(HES_START)
    HES_END_TS = DATA_END_TS
    print('hes читается по частям: проверки покрытия и качества данных пропущены')
print('Найдено названий правил:', len(RULE_MAP), '| не распознаны:', [k for k, v in RULE_MAP.items() if v.startswith('other_')])

## 6. Популяция A: выходы в софт-коллекшн, целевая переменная и базовые (внутренние) признаки
Софт-коллекшн (soft collection) — ранняя стадия взыскания, просрочка 1–30 дней.
* дата скоринга `score_date = dlq_start + OBS_DPD − 1`, начиная с `HES_START + MIN_HISTORY_DAYS`;
* исход должен быть наблюдаемым: `dlq_start ≤ DATA_END − (BAD_DPD − 1)`;
* целевая переменная (таргет, target): `target = 1`, если число дней просрочки (DPD) в эпизоде достигает `BAD_DPD` (DPD ≥ `BAD_DPD`).

**Базовые (внутренние) признаки** (baseline features) используют только внутренние данные банка, известные на дату скоринга:
продукт, канал, сумма, срок с момента выдачи в днях (months on book, признак `bl_mob_days`), доля истекшего срока договора, предыдущие просрочки
по договору и по клиенту (количество, худший DPD на дату скоринга, число дней с момента последней), другие наши договоры
клиента, находящиеся в просрочке сейчас, и число открытых наших договоров.

In [ ]:
POP_START = HES_START_TS + days(MIN_HISTORY_DAYS)
LAST_START = DATA_END_TS - days(BAD_DPD - 1)


def build_collection_population(ep):
    p = ep[(ep['max_dpd'] >= OBS_DPD) & ~ep['overlap']].copy()
    p['score_date'] = p['dlq_start'] + days(OBS_DPD - 1)
    p = p[(p['score_date'] >= POP_START) & (p['dlq_start'] <= LAST_START) & (p['score_date'] <= HES_END_TS)]
    p['target'] = (p['max_dpd'] >= BAD_DPD).astype('int8')
    for name, k in EXTRA_TARGETS.items():
        observed = p['dlq_start'] <= DATA_END_TS - days(k - 1)
        p[name] = np.where(observed, (p['max_dpd'] >= k).astype(float), np.nan)
    p = p.dropna(subset=['iin']).reset_index(drop=True)
    p['ep_id'] = np.arange(len(p))
    p['week'] = p['score_date'].dt.to_period('W-SUN').dt.start_time
    p['hist_days'] = (p['score_date'] - HES_START_TS).dt.days
    return p


def build_baseline(pop, ep_all, cinfo):
    """Внутренние признаки банка на дату скоринга (без данных ПКБ)."""
    b = pop[['ep_id', 'contract', 'iin', 'score_date']].merge(
        cinfo.drop(columns=['iin']), on='contract', how='left')
    out = pd.DataFrame({'ep_id': b['ep_id'].values})
    out['bl_product'] = b['product'].fillna('NA').astype(str).values
    out['bl_channel'] = b['channel'].fillna('NA').astype(str).values
    out['bl_amount'] = b['amount'].values
    mob = (b['score_date'] - b['open_date']).dt.days
    term = (b['end_date'] - b['open_date']).dt.days
    out['bl_mob_days'] = mob.values
    out['bl_term_days'] = term.values
    out['bl_term_elapsed'] = (mob / term.where(term > 0)).values
    hist = ep_all[['contract', 'iin', 'dlq_start', 'max_dpd', 'ep_uid']]
    for lvl, key in (('c', 'contract'), ('i', 'iin')):
        h = pop[['ep_id', key, 'score_date', 'ep_uid']].merge(
            hist[[key, 'dlq_start', 'max_dpd', 'ep_uid']], on=key, suffixes=('', '_h'))
        h = h[(h['dlq_start'] < h['score_date']) & (h['ep_uid_h'] != h['ep_uid'])]
        days_in = (h['score_date'] - h['dlq_start']).dt.days          # DPD, который был у старого эпизода вчера (накануне даты скоринга)
        h['dpd_asof'] = np.minimum(h['max_dpd'], days_in)
        h['end_asof'] = h['dlq_start'] + pd.to_timedelta(h['dpd_asof'] - 1, unit='D')
        h['ongoing'] = h['max_dpd'] >= days_in
        h['recent'] = h['dlq_start'] >= h['score_date'] - days(180)
        g = h.groupby('ep_id')
        agg = pd.DataFrame({
            f'bl_{lvl}_n_prev_ep': g.size(),
            f'bl_{lvl}_n_prev_ep_180d': g['recent'].sum(),
            f'bl_{lvl}_max_prev_dpd': g['dpd_asof'].max(),
            f'bl_{lvl}_days_since_prev': (g['score_date'].first() - g['end_asof'].max()).dt.days})
        if lvl == 'i':
            agg['bl_i_n_dlq_now'] = g['ongoing'].sum()
            agg['bl_i_n_prev_roll'] = (h['dpd_asof'] >= 31).groupby(h['ep_id']).sum()
        agg = agg.reindex(out['ep_id'].values)
        for c in agg.columns:
            vals = agg[c].values.astype(float)
            out[c] = vals if 'days_since' in c else np.nan_to_num(vals, nan=0.0)
    first_dlq = ep_all.groupby('contract')['dlq_start'].min().rename('first_dlq')
    oc = pop[['ep_id', 'iin', 'score_date']].merge(
        cinfo[['contract', 'iin', 'open_date', 'close_date', 'amount', 'from_contracts']].join(first_dlq, on='contract'),
        on='iin')
    known = oc['from_contracts'].astype(bool) | (oc['first_dlq'] <= oc['score_date'])
    oc = oc[known & (oc['open_date'] <= oc['score_date']) & ~(oc['close_date'] < oc['score_date'])]
    g = oc.groupby('ep_id')
    out['bl_i_n_open_contracts'] = g.size().reindex(out['ep_id'].values).fillna(0).values
    out['bl_i_open_amount'] = g['amount'].sum().reindex(out['ep_id'].values).fillna(0).values
    return out.set_index('ep_id')


with Timer('Популяция A и базовые признаки'):
    pop = build_collection_population(episodes)
    base_A = build_baseline(pop, episodes, cinfo)
print(f'Популяция A: {len(pop):,} эпизодов, {pop["iin"].nunique():,} клиентов, '
      f'даты скоринга {pop["score_date"].min().date()} .. {pop["score_date"].max().date()}, '
      f'доля «плохих» (bad rate) {pop["target"].mean():.2%} ({int(pop["target"].sum()):,} «плохих»)')
cohorts = pop.groupby('week').agg(episodes=('target', 'size'), bads=('target', 'sum'), bad_rate=('target', 'mean'),
                                  **{f'{k}_rate': (k, 'mean') for k in EXTRA_TARGETS})
display(cohorts)
if len(pop) == 0:
    raise ValueError('Пустая популяция: проверьте HES_START / MIN_HISTORY_DAYS / DATA_END')

### Популяция B (необязательная): действующие договоры без просрочки, раннее предупреждение (early warning, EWS)
Для каждого договора выбирается один случайный еженедельный срез (snapshot) внутри периода покрытия мониторинга ПКБ
(Первое кредитное бюро, PKB). На дату среза договор должен быть открыт и не находиться в просрочке. `target = 1`
(целевая переменная, таргет), если в течение `EWS_HORIZON` дней начинается эпизод просрочки, в котором число
дней просрочки (DPD) достигает `EWS_MIN_DPD` (DPD ≥ `EWS_MIN_DPD`). Статус «открыт / закрыт» определяется по `close_date`; если в этой колонке
хранится плановая дата окончания договора, то досрочно погашенные договоры остаются в выборке как «хорошие» (good):
это размывает сравнение, но не вносит в него смещения.

In [ ]:
def build_ews_population(cinfo, ep_all, rng):
    last_snap = DATA_END_TS - days(EWS_HORIZON + EWS_MIN_DPD - 2)
    snaps = pd.date_range(POP_START, min(last_snap, HES_END_TS), freq=f'{EWS_SNAPSHOT_FREQ_DAYS}D')
    if len(snaps) == 0:
        return None
    c = cinfo[cinfo['from_contracts'].astype(bool)].dropna(subset=['iin', 'open_date']).copy()   # только полный перечень выданных договоров
    c['score_date'] = snaps[rng.integers(0, len(snaps), len(c))]
    c = c[(c['open_date'] < c['score_date']) & ~(c['end_date'] < c['score_date'])
          & ~((c['closed'] == 1) & (c['close_date'] < c['score_date']))]
    if len(c) > EWS_MAX_ROWS:
        c = c.sample(n=EWS_MAX_ROWS, random_state=SEED)
    m = c[['contract', 'score_date']].merge(ep_all[['contract', 'dlq_start', 'max_dpd']], on='contract')
    dlq_now = (m['dlq_start'] < m['score_date']) & (m['dlq_start'] + pd.to_timedelta(m['max_dpd'] - 1, unit='D') >= m['score_date'] - days(1))
    future = ((m['dlq_start'] >= m['score_date']) & (m['dlq_start'] < m['score_date'] + days(EWS_HORIZON))
              & (m['max_dpd'] >= EWS_MIN_DPD))
    flags = pd.DataFrame({'contract': m['contract'], 'dlq_now': dlq_now, 'future': future}).groupby('contract').max()
    c = c.merge(flags, left_on='contract', right_index=True, how='left')
    c = c[~c['dlq_now'].fillna(False).astype(bool)]
    p = pd.DataFrame({'contract': c['contract'].values, 'iin': c['iin'].values, 'score_date': c['score_date'].values,
                      'target': c['future'].fillna(False).astype('int8').values})
    p['ep_uid'] = -1
    p['ep_id'] = np.arange(len(p))
    p['week'] = p['score_date'].dt.to_period('W-SUN').dt.start_time
    p['hist_days'] = (p['score_date'] - HES_START_TS).dt.days
    return p


pop_B, base_B = None, None
if RUN_EWS and contracts is not None:
    with Timer('Популяция B (раннее предупреждение) и базовые признаки'):
        pop_B = build_ews_population(cinfo, episodes, np.random.default_rng(SEED))
        if pop_B is not None and len(pop_B):
            base_B = build_baseline(pop_B, episodes, cinfo)
    if pop_B is not None and len(pop_B):
        print(f'Популяция B: {len(pop_B):,} срезов договоров, доля «плохих» (bad rate) {pop_B["target"].mean():.2%} '
              f'({int(pop_B["target"].sum()):,} выходов на просрочку)')
else:
    print('Часть B пропущена (нужны RUN_EWS = True и таблица `contracts`)')

## 7. События мониторинга ПКБ (PKB) по клиентам популяции: фильтрация, разбор JSON, отметка наших договоров
Сохраняются только строки по клиентам популяции с `created_date` раньше последней даты скоринга: это сокращает
полную таблицу ещё до разбора JSON. Извлекаемые поля:

| поле | ключи JSON |
|---|---|
| `code` | `contract_code` (сравнивается с нашими `contract_number` / `process_ref`) |
| `cred` | `creditor_type.id` (1 = банк, 2 = МФО, микрофинансовая организация) |
| `v_new`, `v_old` | первое найденное из `overdue_sum`, `debt_sum`, `instalment`, `overdue_instalment_count` |
| `f_new` | первый найденный флаг `is_*` / `has_*` / `stopcredit` (статусы, урегулирование задолженности, рефинансирование) |
| `amt`, `debt`, `outst` | `summa_zaima`, `zadolzhenost`, `outstanding_debt` |
| `prolong_days` | `prolongation_end_date − prolongation_start_date` |

In [ ]:
def prefilter_hes(hes_src, iins, t_to):
    """Строки `hes` по заданным клиентам со временем события < t_to. hes_src: DataFrame или итерируемый набор DataFrame (по частям)."""
    keys = set(pd.unique(pd.Series(list(iins))))
    keys_alt = keys | {k.lstrip('0') for k in keys}            # bin_iin может храниться без ведущих нулей
    chunks = [hes_src] if isinstance(hes_src, pd.DataFrame) else hes_src
    out = []
    for ch in chunks:
        t = ch[EVENT_TIME_COL]
        if not pd.api.types.is_datetime64_any_dtype(t):
            t = to_dt(t)
        idx = np.flatnonzero((t < t_to).values)
        is_num = pd.api.types.is_numeric_dtype(ch[H_IIN])
        raw = ch[H_IIN].values[idx]
        if is_num:
            m = pd.Series(raw).isin({int(k) for k in keys if k.isdigit()}).values
        else:
            probe = pd.Series(raw[:10_000]).astype(str)
            clean = probe.str.fullmatch(r'\d+').fillna(False).astype(bool).mean() > 0.999
            ser = pd.Series(raw) if clean else pd.Series(raw).astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
            m = ser.isin(keys_alt).values
        idx = idx[m]
        cols = [H_IIN, H_RULE, H_OLD, H_NEW] + ([H_ID] if H_ID in ch.columns else [])
        part = ch.iloc[idx][cols].copy()
        part['t'] = t.values[idx]
        out.append(part)
    raw = pd.concat(out, ignore_index=True)
    if H_ID in raw.columns:
        n0 = len(raw)
        raw = raw.drop_duplicates(subset=[H_ID]).drop(columns=[H_ID])
        print(f'  дубликатов по {H_ID} удалено: {n0 - len(raw):,}')
    raw['iin'] = norm_iin(raw[H_IIN]).values
    raw = raw[raw['iin'].isin(keys)].drop(columns=[H_IIN])
    return raw.reset_index(drop=True)


def parse_events(raw, own_codes, chunk=1_000_000):
    names = raw[H_RULE].astype('category')
    rmap = map_rule_names(names.cat.categories)
    RULE_MAP.update(rmap)
    rule = names.cat.rename_categories([rmap[c] for c in names.cat.categories]).astype(str).values
    parts = []
    for i0 in range(0, len(raw), chunk):
        ch = raw.iloc[i0:i0 + chunk]
        nw = list(zip(*[_parse_new(s) for s in ch[H_NEW].values]))
        od = list(zip(*[_parse_old(s) for s in ch[H_OLD].values]))
        parts.append(pd.DataFrame({
            'iin': ch['iin'].values, 't': ch['t'].values, 'rule': rule[i0:i0 + chunk],
            'code': np.array(nw[0], dtype=object), 'cred': np.array(nw[1], dtype='float32'),
            'v_new': np.array(nw[2], dtype=float), 'f_new': np.array(nw[3], dtype='float32'),
            'amt': np.array(nw[4], dtype=float), 'debt': np.array(nw[5], dtype=float),
            'outst': np.array(nw[6], dtype=float), 'prolong_days': np.array(nw[7], dtype='float32'),
            'v_old': np.array(od[0], dtype=float), 'f_old': np.array(od[1], dtype='float32')}))
        print(f'  разобрано {min(i0 + chunk, len(raw)):,} / {len(raw):,}', flush=True)
    ev = pd.concat(parts, ignore_index=True)
    ev['own'] = np.where(ev['code'].isna(), -1, np.where(ev['code'].isin(own_codes), 1, 0)).astype('int8')
    return ev


iins_all = set(pop['iin'])
t_to = pop['score_date'].max()
if pop_B is not None:
    iins_all |= set(pop_B['iin'])
    t_to = max(t_to, pop_B['score_date'].max())
with Timer(f'Фильтрация hes: оставляем {len(iins_all):,} клиентов и события до {t_to.date()}'):
    hes_raw = prefilter_hes(hes, iins_all, t_to)
print(f'{len(hes_raw):,} строк сохранено')
with Timer('Разбор JSON'):
    ev = parse_events(hes_raw, OWN_CODES)
del hes_raw
gc.collect()

ev = ev[ev['rule'] != 'unsubscribe'].reset_index(drop=True)          # техническое правило
ev['fam'] = ev['rule'].map(FAMILY).fillna('unknown')
code_share = ev.groupby('rule')['own'].apply(lambda x: (x >= 0).mean())
CONTRACT_RULES = set(code_share[code_share >= 0.5].index)               # правила, в событиях которых передаётся contract_code
ev['rkey'] = np.where(ev['rule'].isin(CONTRACT_RULES),
                      ev['rule'] + np.where(ev['own'] == 1, '_own', '_oth'), ev['rule'])
ev['rkey'] = ev['rkey'].astype('category')
ev['oth'] = (ev['own'] != 1) & ~ev['rule'].isin(BANK_RULES)
ev = ev.drop(columns=['code'])
own_diag = ev.groupby('rule').agg(events=('own', 'size'), share_with_contract_code=('own', lambda x: (x >= 0).mean()),
                                  share_own_contract=('own', lambda x: (x == 1).mean()))
own_diag.insert(0, 'description', [RULE_DESC.get(c, '') for c in own_diag.index])
print('Сопоставление с нашими договорами по правилам (если для dpd_chg «Доля событий по нашим договорам» '
      'близка к 0, коды договоров не совпали):')
display(own_diag.sort_values('events', ascending=False))

## 8. Признаки ПКБ (PKB)
Для каждого окна наблюдения W (7 / 14 / 30 дней до даты скоринга):
* `n_<rule>_<W>d`: число событий по правилу; правила, в событиях которых есть код договора, разделяются на `_oth`
  (другие кредиторы) и `_own` (наши договоры);
* счётчики по семействам правил (`fam_<family>_oth_n`), общее число событий и число различных сработавших правил
  у других кредиторов;
* значения из JSON: худшее значение дней просрочки (DPD) и число увеличений DPD у других кредиторов, сумма просрочки (максимум,
  увеличения), основной долг (увеличения, уменьшения, чистое изменение), изменение ежемесячного платежа, суммы заявок,
  доля МФО среди заявок, запросов кредитного отчёта и новых займов, дни пролонгации, установка статусов («Лудоман», «Призывник», «Стоп-Кредит», «Банкротство»);
* `days_since_<rule>`: число дней с последнего события по правилу (по всей истории с начала мониторинга).

Группы признаков: `pkb_other` (другие кредиторы: новая для банка информация), `pkb_bank_rule` (правила, заданные
относительно нашего банка), `pkb_own` (наши договоры: повторяют внутренние данные банка, используются только для
проверки на утечку данных (leakage)), `baseline` (базовые, внутренние признаки банка). В таблицах и на графиках они
подписаны как «ПКБ: другие кредиторы», «ПКБ: правила относительно нашего банка», «ПКБ: наши договоры» и
«базовые (внутренние) признаки».

In [ ]:
def rkey_group(rk):
    if rk.endswith('_own'):
        return 'pkb_own'
    base = rk[:-4] if rk.endswith('_oth') else rk
    return 'pkb_bank_rule' if base in BANK_RULES else 'pkb_other'


def rkey_desc(rk):
    base = rk[:-4] if rk.endswith(('_oth', '_own')) else rk
    tail = ' (другие кредиторы)' if rk.endswith('_oth') else ' (наши договоры)' if rk.endswith('_own') else ''
    return RULE_DESC.get(base, base) + tail


def build_pkb_features(pop, ev, windows=WINDOWS, lag_days=EVENT_LAG_DAYS):
    j = pop[['ep_id', 'iin', 'score_date']].merge(ev, on='iin', how='inner')
    cut = j['score_date'] - days(lag_days)
    j = j[(j['t'] < cut).values].copy()
    j['age'] = ((cut.loc[j.index] - j['t']) / np.timedelta64(1, 'D')).astype('float32')
    feats, meta = {}, {}

    def put(name, s, fill, group, family, rule, window, desc):
        feats[name] = (s, fill)
        meta[name] = dict(group=group, family=family, rule=rule, window=window, description=desc)

    for W in windows:
        sub = j[j['age'] <= W]
        cnt = sub.groupby(['ep_id', 'rkey'], observed=True).size().unstack(fill_value=0)
        for rk in cnt.columns:
            base = rk[:-4] if rk.endswith(('_oth', '_own')) else rk
            put(f'n_{rk}_{W}d', cnt[rk], 0, rkey_group(rk), FAMILY.get(base, 'unknown'), base, W,
                f'Число событий: {rkey_desc(rk)}, за последние {W} дн.')
        o = sub[sub['oth']]
        go = o.groupby('ep_id')
        put(f'all_oth_n_{W}d', go.size(), 0, 'pkb_other', 'all', 'all', W, f'Число событий у других кредиторов (все правила), за последние {W} дн.')
        put(f'all_oth_nrules_{W}d', go['rule'].nunique(), 0, 'pkb_other', 'all', 'all', W,
            f'Число различных сработавших правил (другие кредиторы), за последние {W} дн.')
        for fam in ('credit_hunger', 'delinquency', 'status', 'relief', 'positive', 'debt_burden'):
            x = o[o['fam'] == fam]
            put(f'fam_{fam}_oth_n_{W}d', x.groupby('ep_id').size(), 0, 'pkb_other', fam, 'family', W,
                f'Число событий семейства «{RU_VALUES.get(fam, fam)}» (другие кредиторы), за последние {W} дн.')

        def rows(rule, frame=o):
            return frame[frame['rule'] == rule]

        x = rows('dpd_chg')
        put(f'dpd_oth_max_{W}d', x.groupby('ep_id')['v_new'].max(), NAN, 'pkb_other', 'delinquency', 'dpd_chg', W,
            f'Худшее (максимальное) новое значение DPD у других кредиторов, за последние {W} дн.')
        x_up = x[x['v_new'] > x['v_old'].fillna(0)]
        put(f'dpd_oth_n_up_{W}d', x_up.groupby('ep_id').size(), 0, 'pkb_other', 'delinquency', 'dpd_chg', W,
            f'Число увеличений DPD у других кредиторов, за последние {W} дн.')
        put(f'dpd_oth_n_ge30_{W}d', x[x['v_new'] >= 30].groupby('ep_id').size(), 0, 'pkb_other', 'delinquency', 'dpd_chg', W,
            f'Число событий изменения DPD со значением DPD >= 30 у других кредиторов, за последние {W} дн.')
        x = rows('ovd_sum_chg')
        dl = x['v_new'] - x['v_old']
        put(f'ovdsum_oth_max_{W}d', x.groupby('ep_id')['v_new'].max(), NAN, 'pkb_other', 'delinquency', 'ovd_sum_chg', W,
            f'Максимальная сумма просрочки у других кредиторов, за последние {W} дн.')
        put(f'ovdsum_oth_n_up_{W}d', x[dl > 0].groupby('ep_id').size(), 0, 'pkb_other', 'delinquency', 'ovd_sum_chg', W,
            f'Число увеличений суммы просрочки у других кредиторов, за последние {W} дн.')
        put(f'ovdsum_oth_up_sum_{W}d', dl.clip(lower=0).groupby(x['ep_id']).sum(), 0, 'pkb_other', 'delinquency', 'ovd_sum_chg', W,
            f'Суммарный прирост суммы просрочки у других кредиторов, за последние {W} дн.')
        x = rows('debt_chg')
        dl = x['v_new'] - x['v_old']
        put(f'debt_oth_n_up_{W}d', x[dl > 0].groupby('ep_id').size(), 0, 'pkb_other', 'debt_burden', 'debt_chg', W,
            f'Число увеличений основного долга у других кредиторов, за последние {W} дн.')
        put(f'debt_oth_n_down_{W}d', x[dl < 0].groupby('ep_id').size(), 0, 'pkb_other', 'debt_burden', 'debt_chg', W,
            f'Число уменьшений основного долга у других кредиторов, за последние {W} дн.')
        put(f'debt_oth_up_sum_{W}d', dl.clip(lower=0).groupby(x['ep_id']).sum(), 0, 'pkb_other', 'debt_burden', 'debt_chg', W,
            f'Суммарный прирост основного долга у других кредиторов, за последние {W} дн.')
        put(f'debt_oth_net_{W}d', dl.groupby(x['ep_id']).sum(), 0, 'pkb_other', 'debt_burden', 'debt_chg', W,
            f'Чистое изменение основного долга у других кредиторов, за последние {W} дн.')
        x = rows('instalment_chg')
        dl = x['v_new'] - x['v_old']
        put(f'inst_net_{W}d', dl.groupby(x['ep_id']).sum(), 0, 'pkb_other', 'debt_burden', 'instalment_chg', W,
            f'Чистое изменение общего ежемесячного платежа, за последние {W} дн.')
        put(f'inst_n_up_{W}d', x[dl > 0].groupby('ep_id').size(), 0, 'pkb_other', 'debt_burden', 'instalment_chg', W,
            f'Число увеличений ежемесячного платежа, за последние {W} дн.')
        x = rows('app_online')
        put(f'app_amt_sum_{W}d', x.groupby('ep_id')['amt'].sum(), 0, 'pkb_other', 'credit_hunger', 'app_online', W,
            f'Общая сумма онлайн-заявок на кредит, за последние {W} дн.')
        put(f'app_amt_max_{W}d', x.groupby('ep_id')['amt'].max(), NAN, 'pkb_other', 'credit_hunger', 'app_online', W,
            f'Максимальная сумма онлайн-заявки на кредит, за последние {W} дн.')
        put(f'app_n_mfo_{W}d', x[x['cred'] == 2].groupby('ep_id').size(), 0, 'pkb_other', 'credit_hunger', 'app_online', W,
            f'Число онлайн-заявок в МФО, за последние {W} дн.')
        x = rows('report_req')
        put(f'inq_n_mfo_{W}d', x[x['cred'] == 2].groupby('ep_id').size(), 0, 'pkb_other', 'credit_hunger', 'report_req', W,
            f'Число запросов кредитного отчёта со стороны МФО, за последние {W} дн.')
        x = o[o['rule'].isin(['new_loan', 'new_contract', 'new_loan_refin'])]
        put(f'newloan_n_mfo_{W}d', x[x['cred'] == 2].groupby('ep_id').size(), 0, 'pkb_other', 'credit_hunger', 'new_loan', W,
            f'Число новых займов в МФО, за последние {W} дн.')
        put(f'newloan_amt_sum_{W}d', x.groupby('ep_id')['amt'].sum(), 0, 'pkb_other', 'credit_hunger', 'new_loan', W,
            f'Общая сумма новых займов, за последние {W} дн.')
        x = rows('prolong')
        put(f'prolong_days_sum_{W}d', x.groupby('ep_id')['prolong_days'].sum(), 0, 'pkb_other', 'relief', 'prolong', W,
            f'Суммарное число дней предоставленной пролонгации, за последние {W} дн.')
        x = o[o['rule'].isin(STATUS_RULES) & (o['f_new'] == 1)]
        put(f'status_on_n_{W}d', x.groupby('ep_id').size(), 0, 'pkb_other', 'status', 'status', W,
            f'Число установленных статусов («Лудоман», «Призывник», «Стоп-Кредит», «Банкротство»), за последние {W} дн.')
        own = sub[(sub['own'] == 1) & (sub['rule'] == 'dpd_chg')]
        put(f'dpd_own_max_{W}d', own.groupby('ep_id')['v_new'].max(), NAN, 'pkb_own', 'delinquency', 'dpd_chg', W,
            f'Худшее (максимальное) значение DPD по нашим договорам, переданное в ПКБ, за последние {W} дн.')
    rec = j.groupby(['ep_id', 'rkey'], observed=True)['age'].min().unstack()
    for rk in rec.columns:
        base = rk[:-4] if rk.endswith(('_oth', '_own')) else rk
        put(f'days_since_{rk}', rec[rk], NAN, rkey_group(rk), FAMILY.get(base, 'unknown'), base, 'all',
            f'Дней с последнего события: {rkey_desc(rk)}')
    put('days_since_any_oth', j[j['oth']].groupby('ep_id')['age'].min(), NAN, 'pkb_other', 'all', 'all', 'all',
        'Дней с последнего события у других кредиторов')
    idx = pd.Index(pop['ep_id'].values, name='ep_id')
    cols = {}
    for name, (s, fill) in feats.items():
        v = s.reindex(idx)
        if fill == fill:
            v = v.fillna(fill)
        cols[name] = v.values.astype('float32')
    F = pd.DataFrame(cols, index=idx)
    keep = [c for c in F.columns if F[c].nunique(dropna=False) > 1]
    catalog = pd.DataFrame.from_dict(meta, orient='index').loc[keep]
    return F[keep], catalog


with Timer('Признаки ПКБ, популяция A'):
    F_A, catalog = build_pkb_features(pop, ev)
print(f'{F_A.shape[1]} признаков ПКБ | доля эпизодов хотя бы с одним событием ПКБ до даты скоринга: '
      f'{F_A["days_since_any_oth"].notna().mean():.1%}')
display(catalog.groupby(['group', 'family']).size().unstack(fill_value=0))

## 9. Биннинг, WoE и IV
Числовые признаки: до 20 квантильных бинов (интервалов) или бинов по отдельным значениям; малые бины
(< `MIN_BIN_SHARE`) присоединяются к соседнему бину с наиболее близкой долей «плохих» (bad rate); затем соседние бины
объединяются, пока доля «плохих» не станет монотонной, а затем — до тех пор, пока бинов не станет `MAX_BINS` или меньше.
Пропущенные значения образуют отдельный бин. Категориальные признаки: один бин на категорию, редкие категории
объединяются. WoE (вес свидетельства) и информационная ценность (IV): `WoE = ln(%good / %bad)` со сглаживанием 0,5,
`IV = Σ (%good − %bad) · WoE`, где %good / %bad — доля «хороших» / «плохих» в бине от их общего числа.
Джини (Gini) и KS рассчитываются для признака после биннинга. Признак помечается как *ненадёжный* (unreliable), если
вне его самого крупного бина находится меньше `MIN_BADS_RELIABLE` «плохих» наблюдений (типично для редких событий).

In [ ]:
# Форматирование границы бина для подписи интервала (целые и большие значения выводятся без дробной части).
def _fmt(v):
    if not np.isfinite(v):
        return 'inf'
    return f'{v:,.0f}' if abs(v - round(v)) < 1e-9 or abs(v) >= 100 else f'{v:,.3g}'


# Подбор бинов (интервалов) для одного признака по целевой переменной y.
# Категориальный признак: свой бин для каждой категории с долей >= min_share, остальные -> 'OTHER', пропуски -> 'MISSING'.
# Числовой признак: стартовые бины по квантилям, затем слияние мелких бинов, монотонизация доли «плохих» (bad rate)
# и сокращение числа бинов до max_bins. Последний код всегда отведён под пропуски ('MISSING').
def fit_bins(x, y, max_bins=MAX_BINS, min_share=MIN_BIN_SHARE, monotonic=True, n_init=20):
    x = pd.Series(x, copy=False)
    y = np.asarray(y, dtype=float)
    N = len(y)
    if not pd.api.types.is_numeric_dtype(x) or pd.api.types.is_bool_dtype(x):
        s = x.astype(object)
        s = s.where(s.notna(), '__MISSING__').astype(str)
        vc = s.value_counts()
        cats = [c for c in vc.index if c != '__MISSING__' and vc[c] >= min_share * N]
        return {'kind': 'cat', 'cats': cats, 'n_codes': len(cats) + 2,
                'labels': cats + ['OTHER', 'MISSING']}
    # числовой признак: пропуски выделяются в отдельный бин
    xv = x.values.astype(float)
    miss = np.isnan(xv)
    v, yv = xv[~miss], y[~miss]
    if len(v) == 0:
        return {'kind': 'num', 'edges': np.array([]), 'n_codes': 2, 'labels': ['all', 'MISSING']}
    # стартовые границы: все уникальные значения (если их не больше n_init) или n_init квантилей
    u = np.unique(v)
    if len(u) <= n_init:
        edges = u[:-1]
    else:
        edges = np.unique(np.quantile(v, np.linspace(0, 1, n_init + 1)[1:-1]))
        edges = edges[edges < u[-1]]
    k = len(edges) + 1
    code = np.searchsorted(edges, v, side='left')
    n = list(np.bincount(code, minlength=k).astype(float))
    b = list(np.bincount(code, weights=yv, minlength=k))
    hi = list(edges) + [np.inf]

    def merge(i):                      # объединить бин i с бином i + 1
        n[i] += n[i + 1]; b[i] += b[i + 1]; hi[i] = hi[i + 1]
        del n[i + 1], b[i + 1], hi[i + 1]

    def br(i):                         # доля «плохих» (bad rate) в бине i
        return b[i] / n[i] if n[i] > 0 else 0.0

    # 1) бины с долей меньше min_share сливаются с соседним бином с наиболее близкой долей «плохих»
    min_n = max(min_share * N, 1)
    while len(n) > 1:
        i = int(np.argmin(n))
        if n[i] >= min_n:
            break
        if i == 0:
            merge(0)
        elif i == len(n) - 1:
            merge(i - 1)
        elif abs(br(i) - br(i - 1)) <= abs(br(i) - br(i + 1)):
            merge(i - 1)
        else:
            merge(i)
    # 2) монотонность: направление тренда доли «плохих» берётся из взвешенной линейной регрессии,
    #    соседние бины, нарушающие это направление, сливаются
    if monotonic and len(n) > 2:
        rates = np.array([br(i) for i in range(len(n))])
        sign = np.sign(np.polyfit(np.arange(len(n)), rates, 1, w=np.sqrt(np.array(n)))[0]) or 1.0
        changed = True
        while changed and len(n) > 1:
            changed = False
            for i in range(len(n) - 1):
                if sign * (br(i + 1) - br(i)) < 0:
                    merge(i)
                    changed = True
                    break
    # 3) пока бинов больше max_bins, сливается пара соседних бинов с наиболее близкой долей «плохих»
    while len(n) > max_bins:
        merge(int(np.argmin([abs(br(i + 1) - br(i)) for i in range(len(n) - 1)])))
    edges = np.array(hi[:-1], dtype=float)
    # подписи интервалов: '<= a', '(a, b]', '> b' (служебные метки 'all', 'MISSING' остаются кодами; на русский их переводит ru_labels при показе)
    labels = []
    lo = -np.inf
    for h in hi:
        labels.append(f'<= {_fmt(h)}' if lo == -np.inf and np.isfinite(h) else
                      f'> {_fmt(lo)}' if not np.isfinite(h) and lo != -np.inf else
                      'all' if not np.isfinite(h) else f'({_fmt(lo)}, {_fmt(h)}]')
        lo = h
    return {'kind': 'num', 'edges': edges, 'n_codes': len(edges) + 2, 'labels': labels + ['MISSING']}


# Присвоение кода бина каждому значению по уже подобранной модели бинов.
# Категории: номер категории, k = 'OTHER', k + 1 = 'MISSING'; числа: номер интервала, пропуски — последний код.
def apply_bins(x, model):
    if model['kind'] == 'cat':
        s = pd.Series(x, copy=False).astype(object)
        lut = {c: i for i, c in enumerate(model['cats'])}
        k = len(model['cats'])
        codes = s.astype(str).map(lut).fillna(k).values.astype(int)
        codes[s.isna().values] = k + 1
        return codes
    xv = np.asarray(x, dtype=float)
    codes = np.searchsorted(model['edges'], xv, side='left')
    codes[np.isnan(xv)] = len(model['edges']) + 1
    return codes


# Статистики по бинам: число наблюдений n, «плохих» b, «хороших» g, WoE и вклад бина в IV.
# WoE = ln(доля «хороших» / доля «плохих») со сглаживанием +0.5, чтобы бин без «плохих» не давал бесконечность.
def bin_stats(codes, y, n_codes):
    y = np.asarray(y, dtype=float)
    n = np.bincount(codes, minlength=n_codes).astype(float)
    b = np.bincount(codes, weights=y, minlength=n_codes)
    g = n - b
    ne = n > 0
    k = max(int(ne.sum()), 1)
    B, G = b.sum(), g.sum()
    db = (b + 0.5) / (B + 0.5 * k)
    dg = (g + 0.5) / (G + 0.5 * k)
    woe = np.where(ne, np.log(dg / db), 0.0)
    iv = np.where(ne, (dg - db) * woe, 0.0)
    return n, b, g, woe, iv


def binned_auc(n, b, risk):
    """AUC скора по бинам; risk: одно значение на бин, больше = рискованнее (совпадающие значения учитываются)."""
    g = n - b
    B, G = b.sum(), g.sum()
    if B == 0 or G == 0:
        return NAN
    o = np.argsort(risk, kind='mergesort')
    bs, gs, rs = b[o], g[o], np.asarray(risk)[o]
    auc, cum_g, i = 0.0, 0.0, 0
    while i < len(rs):
        j = i
        while j + 1 < len(rs) and rs[j + 1] == rs[i]:
            j += 1
        bb, gg = bs[i:j + 1].sum(), gs[i:j + 1].sum()
        auc += bb * (cum_g + 0.5 * gg)
        cum_g += gg
        i = j + 1
    return auc / (B * G)


# Статистика Колмогорова–Смирнова (KS) для скора по бинам: максимум расхождения накопленных долей «плохих» и «хороших».
def binned_ks(n, b, risk):
    g = n - b
    if b.sum() == 0 or g.sum() == 0:
        return NAN
    o = np.argsort(-np.asarray(risk), kind='mergesort')
    return float(np.max(np.abs(np.cumsum(b[o]) / b.sum() - np.cumsum(g[o]) / g.sum())))


# Таблица бинов одного признака: подпись бина, n, число «плохих», доля бина, доля «плохих», WoE, IV (пустые бины скрыты).
def bin_table(x, y, model):
    codes = apply_bins(x, model)
    n, b, g, woe, iv = bin_stats(codes, y, model['n_codes'])
    t = pd.DataFrame({'bin': model['labels'], 'n': n, 'bads': b, 'share': n / n.sum(),
                      'bad_rate': np.where(n > 0, b / np.maximum(n, 1), np.nan), 'woe': woe, 'iv': iv})
    return t[t['n'] > 0]


# Сводные метрики одного признака: IV, Джини (Gini) и KS (скор = -WoE), число непустых бинов,
# число «плохих» вне самого крупного бина (мера надёжности) и направление зависимости риска от значения признака.
def feature_iv(x, y, model=None, **kw):
    model = model or fit_bins(x, y, **kw)
    codes = apply_bins(x, model)
    n, b, g, woe, iv = bin_stats(codes, y, model['n_codes'])
    ne = n > 0
    risk = -woe
    big = int(np.argmax(n))
    rate = np.where(ne, b / np.maximum(n, 1), np.nan)
    num_bins = [i for i in range(model['n_codes'] - 1) if ne[i]] if model['kind'] == 'num' else []
    direction = ''
    if len(num_bins) >= 2:
        direction = 'risk up with value' if rate[num_bins[-1]] > rate[num_bins[0]] else 'risk down with value'
    return model, dict(iv=float(iv.sum()), gini=2 * binned_auc(n[ne], b[ne], risk[ne]) - 1,
                       ks=binned_ks(n[ne], b[ne], risk[ne]), n_bins=int(ne.sum()),
                       bads_outside_main_bin=float(b.sum() - b[big]), direction=direction)


# Таблица IV по списку признаков: метрики feature_iv + доли пропусков и ненулевых значений,
# класс силы по IV (strength), флаг надёжности (reliable) и группа признака из каталога (по умолчанию 'baseline').
def compute_iv(X, y, features, catalog=None):
    y = np.asarray(y, dtype=float)
    rows, models = [], {}
    for f in features:
        x = X[f]
        m, r = feature_iv(x, y)
        models[f] = m
        xv = pd.to_numeric(x, errors='coerce') if m['kind'] == 'num' else None
        r.update(feature=f,
                 share_missing=float(pd.isna(x).mean()),
                 share_nonzero=float(((xv != 0) & xv.notna()).mean()) if xv is not None else NAN)
        rows.append(r)
    if not rows:
        return pd.DataFrame(columns=['iv', 'gini', 'ks', 'n_bins', 'bads_outside_main_bin', 'direction', 'share_missing',
                                     'share_nonzero', 'strength', 'reliable', 'group']), models
    t = pd.DataFrame(rows).set_index('feature')
    t['strength'] = t['iv'].map(iv_strength)
    t['reliable'] = t['bads_outside_main_bin'] >= MIN_BADS_RELIABLE
    if catalog is not None:
        t = t.join(catalog, how='left')
    t['group'] = t['group'].fillna('baseline') if 'group' in t else 'baseline'
    return t.sort_values('iv', ascending=False), models


# PSI (индекс стабильности популяции) между двумя распределениями по бинам; eps защищает от log(0).
def psi(expected, actual, eps=1e-4):
    e = np.asarray(expected, float) / max(np.sum(expected), 1)
    a = np.asarray(actual, float) / max(np.sum(actual), 1)
    e, a = np.clip(e, eps, None), np.clip(a, eps, None)
    return float(np.sum((a - e) * np.log(a / e)))

## 10. Результаты A1: какие правила важны (флаги событий)
Для каждого правила мониторинга ПКБ (Первое кредитное бюро, PKB) — с разбивкой на другие кредиторы / наши договоры там,
где в событии есть код договора, — флаг «хотя бы одно событие за последние `IMPACT_WINDOW` дней» сравнивается с общей
долей «плохих» (bad rate). **Лифт (lift)** = доля «плохих» среди эпизодов с флагом / общая доля «плохих». Эту таблицу
стоит обсуждать с подразделением взыскания (collection): как часто происходит событие и насколько сильно оно меняет долю
перекатов (roll rate).

In [ ]:
# Целевая переменная (таргет) популяции A в порядке строк матрицы признаков F_A и общая доля «плохих».
y_A = pop.set_index('ep_id').loc[F_A.index, 'target'].values.astype(float)
BR_A = y_A.mean()


# Влияние каждого правила: флаг «было хотя бы одно событие за окно window» против доли «плохих».
# Используются только признаки-счётчики n_<правило>_<window>d; лифт = доля «плохих» с флагом / общая доля «плохих».
# reliable: среди эпизодов с флагом не меньше MIN_BADS_RELIABLE / 2 «плохих» и флаг стоит не у всех эпизодов.
def rule_impact(F, y, catalog, window=IMPACT_WINDOW):
    y = np.asarray(y, float)
    rows = []
    br = y.mean()
    for f, meta in catalog.iterrows():
        if not (f.startswith('n_') and meta['window'] == window):
            continue
        flag = (F[f].values > 0)
        nf = int(flag.sum())
        if nf == 0:
            continue
        bf = float(y[flag].sum())
        n_, b_, g_, woe, iv = bin_stats(flag.astype(int), y, 2)
        rows.append(dict(rule_key=f[2:-(len(str(window)) + 2)], group=meta['group'], family=meta['family'],
                         description=rkey_desc(f[2:-(len(str(window)) + 2)]),
                         share_flagged=nf / len(y), n_flagged=nf, bads_flagged=int(bf),
                         bad_rate_flagged=bf / nf, bad_rate_not=float(y[~flag].mean()) if nf < len(y) else NAN,
                         lift=(bf / nf) / br if br > 0 else NAN, iv_flag=float(iv.sum()),
                         reliable=bf >= MIN_BADS_RELIABLE / 2 and (len(y) - nf) > 0))
    if not rows:
        return pd.DataFrame(columns=['rule_key', 'group', 'family', 'description', 'share_flagged', 'n_flagged',
                                     'bads_flagged', 'bad_rate_flagged', 'bad_rate_not', 'lift', 'iv_flag', 'reliable'])
    return pd.DataFrame(rows).sort_values('iv_flag', ascending=False).reset_index(drop=True)


impact_A = rule_impact(F_A, y_A, catalog)
display(impact_A)


# График лифта по правилам: красный — событие повышает риск, синий — снижает;
# правила, флаг которых стоит менее чем у min_share эпизодов, не показываются.
def plot_rule_lift(impact, title, min_share=0.001):
    d = impact[impact['share_flagged'] >= min_share].sort_values('lift')
    if d.empty:
        return
    fig, ax = plt.subplots(figsize=(9, 0.32 * len(d) + 1.2))
    col = np.where(d['lift'] >= 1, PAL['red'], PAL['blue'])
    ax.barh(d['rule_key'], d['lift'] - 1, left=1, color=col, height=0.6)
    ax.axvline(1, color=PAL['muted'], lw=0.8)
    for i, (l, s, rel) in enumerate(zip(d['lift'], d['share_flagged'], d['reliable'])):
        ax.text(max(l, 1) + 0.03, i, f'{s:.1%} с событием' + ('' if rel else ', мало «плохих»'), va='center',
                fontsize=7, color=PAL['muted'])
    ax.set_xlabel('лифт = доля «плохих» при наличии события / общая доля «плохих»')
    ax.set_title(title)
    ax.grid(axis='y', visible=False)
    plt.tight_layout(); plt.show()


plot_rule_lift(impact_A, f'Популяция A: перекат в DPD {BAD_DPD}+, событие за последние {IMPACT_WINDOW} дней '
                         '(красный — риск выше, синий — риск ниже)')

## 11. Результаты A2: информационная ценность (IV) каждого признака
Все признаки ПКБ (Первое кредитное бюро, PKB) и базовые (внутренние) признаки банка на полной популяции A.

In [ ]:
# Матрица признаков популяции A: признаки ПКБ + базовые (внутренние) признаки.
X_A = F_A.join(base_A)
features_A = list(X_A.columns)
with Timer(f'Расчёт IV для {len(features_A)} признаков'):
    iv_A, models_A = compute_iv(X_A, y_A, features_A, catalog)
# колонки, выводимые в таблице IV
cols_show = ['group', 'family', 'window', 'iv', 'strength', 'gini', 'ks', 'n_bins', 'share_nonzero', 'share_missing',
             'bads_outside_main_bin', 'reliable', 'direction', 'description']
print('Топ-40 признаков по IV:')
display(iv_A[cols_show].head(40))
print('IV по группам (число признаков в каждом классе силы IV):')
display(pd.crosstab(iv_A['group'], iv_A['strength']))


# Горизонтальная диаграмма IV топ-признаков: цвет столбца = группа признака,
# вертикальные линии — пороги силы IV (0.02 слабый, 0.1 средний, 0.3 сильный).
def plot_iv(iv, title, top=30):
    d = iv.head(top).iloc[::-1]
    fig, ax = plt.subplots(figsize=(9, 0.27 * len(d) + 1.2))
    ax.barh(d.index, d['iv'], color=[GROUP_COLOR.get(g, PAL['gray']) for g in d['group']], height=0.62)
    for thr in (0.02, 0.1, 0.3):
        ax.axvline(thr, color='#c9c8c2', lw=0.8)
    groups = [g for g in GROUP_COLOR if g in set(d['group'])]
    ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=GROUP_COLOR[g]) for g in groups],
              labels=[RU_VALUES.get(g, g) for g in groups], loc='lower right')
    ax.set_xlabel('Информационная ценность, IV (линии на уровнях 0.02 / 0.1 / 0.3)')
    ax.set_title(title)
    ax.grid(axis='y', visible=False)
    plt.tight_layout(); plt.show()


plot_iv(iv_A, 'Популяция A: топ признаков по IV')
pkb_groups = ('pkb_other', 'pkb_bank_rule')
plot_iv(iv_A[iv_A['group'].isin(pkb_groups)], 'Популяция A: топ признаков ПКБ (другие кредиторы и правила банка)')

### Профили WoE (вес свидетельства) самых сильных признаков ПКБ (Первое кредитное бюро, PKB)
Столбцы: доля «плохих» (bad rate) в каждом бине (интервале); над каждым столбцом — доля бина в выборке. Горизонтальная
линия: общая доля «плохих».

In [ ]:
# Сетка графиков: доля «плохих» по бинам для каждого признака (подпись над столбцом — доля бина,
# горизонтальная линия — общая доля «плохих», в заголовке — IV признака).
def plot_woe_grid(X, y, models, feats, ncols=3, title=''):
    feats = list(feats)
    if not feats:
        return
    nrows = int(math.ceil(len(feats) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.1 * ncols, 2.9 * nrows), squeeze=False)
    br = np.mean(y)
    for ax, f in zip(axes.ravel(), feats):
        t = bin_table(X[f], y, models[f])
        ax.bar(range(len(t)), t['bad_rate'], color=PAL['blue'], width=0.6)
        ax.axhline(br, color=PAL['muted'], lw=0.8)
        for i, s in enumerate(t['share']):
            ax.text(i, t['bad_rate'].iloc[i], f'{s:.0%}', ha='center', va='bottom', fontsize=7, color=PAL['muted'])
        ax.set_xticks(range(len(t)))
        ax.set_xticklabels(ru_labels(t['bin']), rotation=30, ha='right', fontsize=7)
        ax.set_title(f'{f}\nIV {bin_table(X[f], y, models[f])["iv"].sum():.3f}', fontsize=8.5)
        ax.grid(axis='x', visible=False)
    for ax in axes.ravel()[len(feats):]:
        ax.axis('off')
    fig.suptitle(title, fontsize=10.5, fontweight='bold')
    plt.tight_layout(); plt.show()


# до 12 самых сильных по IV надёжных признаков ПКБ
top_pkb = iv_A[iv_A['group'].isin(pkb_groups) & iv_A['reliable']].head(12).index
plot_woe_grid(X_A, y_A, models_A, top_pkb, title='Доля «плохих» по бинам: самые сильные признаки ПКБ')
print('Таблицы бинов для топ-5 признаков ПКБ:')
for f in top_pkb[:5]:
    print(f'\n{f}: {catalog.loc[f, "description"]}')
    display(bin_table(X_A[f], y_A, models_A[f]))

## 12. Стабильность
* **Понедельные IV и Джини (Gini)** при бинах, зафиксированных на полной выборке; Джини за неделю считается по порядку
  WoE (вес свидетельства) полной выборки, поэтому смена направления зависимости проявляется как низкое или отрицательное
  значение.
* **PSI** (индекс стабильности популяции) распределения по бинам каждой недели относительно первой недели.
* **Полная 30-дневная история**: у 30-дневных признаков эпизодов с датой скоринга в первые недели мониторинга окно
  наблюдения покрывает меньше 30 дней событий; их IV пересчитывается на эпизодах с полной историей.
* **IV в разрезе продуктов**.

In [ ]:
# Понедельная стабильность при бинах, зафиксированных на полной выборке: IV и Джини (Gini) по неделям,
# PSI распределения по бинам относительно первой недели. Признак считается стабильным (stable),
# если минимальный понедельный Джини > 0 и максимальный PSI < 0.25.
def weekly_stability(X, y, week, models, feats):
    y = np.asarray(y, float)
    wk = np.sort(pd.unique(week))
    rows = []
    for f in feats:
        m = models[f]
        codes = apply_bins(X[f], m)
        n_all, b_all, _, woe_all, _ = bin_stats(codes, y, m['n_codes'])
        ref = None
        ivs, ginis, psis = [], [], []
        for w in wk:
            mk = (week == w)
            n, b, g, woe, iv = bin_stats(codes[mk], y[mk], m['n_codes'])
            ne = n > 0
            ivs.append(iv.sum())
            ginis.append(2 * binned_auc(n[ne], b[ne], -woe_all[ne]) - 1)
            if ref is None:
                ref = n
            psis.append(psi(ref, n))
        rows.append(dict(feature=f, iv_min=np.min(ivs), iv_max=np.max(ivs), iv_cv=np.std(ivs) / max(np.mean(ivs), 1e-9),
                         gini_min_week=np.nanmin(ginis), gini_max_week=np.nanmax(ginis), psi_max=np.max(psis),
                         **{f'iv_{pd.Timestamp(w):%d.%m}': v for w, v in zip(wk, ivs)}))
    if not rows:
        return pd.DataFrame(columns=['stable'])
    t = pd.DataFrame(rows).set_index('feature')
    t['stable'] = (t['gini_min_week'] > 0) & (t['psi_max'] < 0.25)
    return t


# признаки для проверки стабильности: IV >= IV_SELECT и надёжные (не более 40)
stab_feats = list(iv_A[(iv_A['iv'] >= IV_SELECT) & iv_A['reliable']].index[:40])
week_A = pop.set_index('ep_id').loc[X_A.index, 'week'].values
stab_A = weekly_stability(X_A, y_A, week_A, models_A, stab_feats)
display(iv_A.loc[stab_feats, ['group', 'iv']].join(stab_A))

# график понедельной IV для 6 сильнейших признаков ПКБ
top6 = [f for f in iv_A[iv_A['group'].isin(pkb_groups)].index if f in stab_A.index][:6]
if top6:
    wcols = [c for c in stab_A.columns if c.startswith('iv_') and c[3:5].isdigit()]
    fig, ax = plt.subplots(figsize=(9, 3))
    for i, f in enumerate(top6):
        ax.plot([c[3:] for c in wcols], stab_A.loc[f, wcols].values, color=SERIES[i], marker='o', ms=4, label=f)
    ax.set_title('Понедельная IV топ-признаков ПКБ (бины зафиксированы на полной выборке)')
    ax.set_xlabel('неделя скоринга (начало)')
    ax.set_ylabel('IV')
    ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1))
    plt.tight_layout(); plt.show()

# признаки с окном 30 дней: IV на всех эпизодах против IV только на эпизодах с полностью заполненным окном наблюдения
full_hist = pop.set_index('ep_id').loc[X_A.index, 'hist_days'].values >= max(WINDOWS)
f30 = [f for f in iv_A.index if catalog.reindex([f])['window'].iloc[0] == max(WINDOWS)]
if full_hist.sum() > 0 and full_hist.mean() < 1 and f30:
    y_fh = y_A[full_hist]
    iv_fh = compute_iv(X_A.loc[full_hist], y_fh, f30)[0]['iv'] if y_fh.sum() > 0 else pd.Series(dtype=float)
    comp = pd.DataFrame({'iv_all_episodes': iv_A.loc[f30, 'iv'], 'iv_full_30d_history': iv_fh}).sort_values('iv_all_episodes', ascending=False)
    print(f'{max(WINDOWS)}-дневные признаки: у {full_hist.sum():,} эпизодов ({full_hist.mean():.0%}) окно наблюдения заполнено полностью')
    display(comp.head(20))

# IV топ-15 признаков ПКБ по продуктам (бины с полной выборки; продукты, где меньше 20 «плохих», пропускаются)
prod_A = base_A.loc[X_A.index, 'bl_product'].values
top_iv_feats = list(iv_A[iv_A['group'].isin(pkb_groups)].head(15).index)
by_prod = {}
for p_ in pd.Series(prod_A).value_counts().index[:6]:
    mk = prod_A == p_
    if y_A[mk].sum() < 20:
        continue
    by_prod[f'{p_} (n={mk.sum():,}, доля «плохих» {y_A[mk].mean():.1%})'] = {
        f: bin_stats(apply_bins(X_A.loc[mk, f], models_A[f]), y_A[mk], models_A[f]['n_codes'])[4].sum() for f in top_iv_feats}
print('IV топ-признаков ПКБ в разрезе продуктов (бины построены на полной выборке):')
display(pd.DataFrame(by_prod))

## 13. Короткий список признаков (short-list): фильтр по корреляции
Кандидаты: признаки ПКБ (Первое кредитное бюро, PKB) групп `pkb_other` (другие кредиторы) и `pkb_bank_rule` (правила
банка) с информационной ценностью (IV) ≥ `IV_SELECT`, надёжные (reliable) и стабильные.
Жадный отбор в порядке убывания IV: признак пропускается, если модуль его ранговой корреляции Спирмена (Spearman)
с уже отобранным признаком превышает `CORR_MAX`.

In [ ]:
# Ранговая корреляция Спирмена между признаками. Пропуск означает «события не было», поэтому перед ранжированием
# он заменяется: у признаков days_since_* (дней с последнего события) — на очень большое число, у остальных — на -1.
def spearman(X, feats):
    """Корреляция Спирмена; пропуск означает «события не было»: +inf для days_since_*, -1 для остальных признаков."""
    D = X[feats].astype(float)
    for f in feats:
        D[f] = D[f].fillna(1e9 if f.startswith('days_since') else -1)
    return D.rank().corr()


# Жадный отбор: признаки перебираются в порядке убывания IV; признак добавляется, только если его корреляция
# (по модулю) с каждым уже отобранным признаком меньше corr_max. Отбор останавливается на max_n признаках.
def greedy_select(iv_series, corr, corr_max=CORR_MAX, max_n=MAX_MODEL_FEATURES):
    sel = []
    for f in iv_series.index:
        if all(abs(corr.loc[f, s]) < corr_max for s in sel):
            sel.append(f)
        if len(sel) >= max_n:
            break
    return sel


# кандидаты: признаки ПКБ (другие кредиторы и правила банка) с IV >= IV_SELECT, надёжные (reliable)
cand = iv_A[iv_A['group'].isin(pkb_groups) & (iv_A['iv'] >= IV_SELECT) & iv_A['reliable']]
# остаются только стабильные признаки (признак, не попавший в проверку стабильности, считается стабильным)
cand = cand[[stab_A['stable'].get(f, True) for f in cand.index]]
if len(cand):
    # корреляция и отбор — среди 60 кандидатов с наибольшей IV
    corr_A = spearman(X_A, list(cand.index[:60]))
    shortlist = greedy_select(cand['iv'].iloc[:60], corr_A)
    print(f'{len(cand)} — число кандидатов, {len(shortlist)} — размер короткого списка:')
    display(iv_A.loc[shortlist, ['group', 'family', 'window', 'iv', 'gini', 'share_nonzero', 'description']])
    # тепловая карта корреляций для первых 15 признаков короткого списка
    sl = shortlist[:15]
    fig, ax = plt.subplots(figsize=(0.45 * len(sl) + 3, 0.45 * len(sl) + 2))
    im = ax.imshow(corr_A.loc[sl, sl].values, cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_xticks(range(len(sl))); ax.set_xticklabels(sl, rotation=60, ha='right', fontsize=7)
    ax.set_yticks(range(len(sl))); ax.set_yticklabels(sl, fontsize=7)
    ax.grid(False)
    fig.colorbar(im, ax=ax, shrink=0.7, label='Корреляция Спирмена')
    ax.set_title('Корреляция признаков ПКБ из короткого списка')
    plt.tight_layout(); plt.show()
else:
    shortlist = []
    print('Ни один признак ПКБ не прошёл фильтры по IV / надёжности / стабильности')

## 14. Прирост качества (инкрементальный эффект) сверх внутренней базовой модели (вне времени, out-of-time, OOT)
* разбиение по дате скоринга: эпизоды с более ранними датами (доля `TRAIN_SHARE`) — обучающая выборка (train),
  остальные — тестовая выборка (test), т. е. проверка вне времени;
* бины (интервалы), информационная ценность (IV) и отбор признаков заново подбираются **только на обучающей выборке**;
* **Модель A**: логистическая регрессия (logistic regression) на WoE (вес свидетельства) базовых (внутренних) признаков;
  **Модель B** = A + отобранные признаки ПКБ (Первое кредитное бюро, PKB) групп `pkb_other` и `pkb_bank_rule`;
  **Модель C** = B + события ПКБ по нашим договорам (показывает, что добавляет «сырой» поток событий; в основном он
  дублирует внутренние данные банка);
* ΔДжини (Gini) на тестовой выборке с бутстрэп-оценкой 95%-го доверительного интервала (ДИ); ресэмплинг выполняется по клиентам, чтобы
  несколько договоров одного клиента попадали в выборку вместе;
* контрольная проверка на градиентном бустинге (gradient boosting) по всем признакам (без отбора).

In [ ]:
# Градиентный бустинг на гистограммах; в старых версиях sklearn нужен экспериментальный импорт.
# Если класс недоступен, контрольная проверка на бустинге пропускается.
try:
    from sklearn.ensemble import HistGradientBoostingClassifier
except ImportError:
    try:
        from sklearn.experimental import enable_hist_gradient_boosting  # noqa: F401
        from sklearn.ensemble import HistGradientBoostingClassifier
    except ImportError:
        HistGradientBoostingClassifier = None


# Джини (Gini) = 2 * AUC - 1 (с необязательными весами наблюдений)
def gini(y, s, w=None):
    return 2 * roc_auc_score(y, s, sample_weight=w) - 1


# Доля всех «плохих», попавших в верхнюю долю q наблюдений с наибольшим скором (capture rate)
def capture(y, s, q):
    o = np.argsort(-np.asarray(s), kind='mergesort')
    k = max(int(np.ceil(q * len(y))), 1)
    return float(np.asarray(y)[o][:k].sum() / max(np.sum(y), 1))


# Бутстрэп разницы Джини (модель s_b минус модель s_a): ресэмплинг кластеров (клиентов по ИИН), чтобы все договоры
# одного клиента попадали в выборку вместе. Возвращает 2.5%, 50% и 97.5% перцентили (границы 95% ДИ и медиану).
def boot_delta(y, s_a, s_b, clusters, n_boot=N_BOOT, seed=SEED):
    codes, uniq = pd.factorize(pd.Series(clusters))
    nc = len(uniq)
    r = np.random.default_rng(seed)
    out = []
    for _ in range(n_boot):
        # вес наблюдения = сколько раз его клиент попал в бутстрэп-выборку
        w = np.bincount(r.integers(0, nc, nc), minlength=nc)[codes].astype(float)
        mk = w > 0
        if len(np.unique(y[mk])) < 2:
            continue
        out.append(gini(y[mk], s_b[mk], w[mk]) - gini(y[mk], s_a[mk], w[mk]))
    return np.percentile(out, [2.5, 50, 97.5]) if out else np.array([NAN] * 3)


# Отбор признаков только на обучающей выборке: IV >= iv_min и надёжность, затем жадный фильтр по корреляции
# (только для числовых признаков) в порядке убывания IV; не более max_n признаков.
# Возвращает (список отобранных признаков, таблица IV на обучающей выборке).
def select_on_train(Xtr, ytr, feats, max_n=MAX_MODEL_FEATURES, iv_min=IV_SELECT):
    if not feats:
        return [], pd.DataFrame()
    t, _ = compute_iv(Xtr, ytr, feats)
    t = t[(t['iv'] >= iv_min) & t['reliable']]
    if t.empty:
        return [], t
    num = [f for f in t.index[:60] if pd.api.types.is_numeric_dtype(Xtr[f])]
    corr = spearman(Xtr, num) if num else None
    sel = []
    for f in t.index[:60]:
        if corr is not None and f in num and any(s in num and abs(corr.loc[f, s]) >= CORR_MAX for s in sel):
            continue
        sel.append(f)
        if len(sel) >= max_n:
            break
    return sel, t


# Бины и WoE, подобранные на обучающей выборке: {признак: (модель бинов, значения WoE по кодам бинов)}
def woe_fit(Xtr, ytr, feats):
    enc = {}
    for f in feats:
        m = fit_bins(Xtr[f], ytr)
        enc[f] = (m, bin_stats(apply_bins(Xtr[f], m), ytr, m['n_codes'])[3])
    return enc


# Перевод признаков в WoE по ранее подобранным бинам (строки — наблюдения, столбцы — признаки)
def woe_apply(X, enc, feats):
    return np.column_stack([enc[f][1][apply_bins(X[f], enc[f][0])] for f in feats]) if feats else np.zeros((len(X), 0))


# Оценка прироста качества: разбиение вне времени, модели A / B / C (логистическая регрессия на WoE),
# Джини и доля выявленных «плохих» в топ-10/20/30% на тестовой выборке, бутстрэп-ДИ для ΔДжини,
# контрольная проверка на градиентном бустинге и график кумулятивной доли выявленных «плохих» (gains).
# clusters — ИИН клиентов (для бутстрэпа по клиентам), label — подпись популяции в выводе и на графике.
def incremental_value(X, y, score_date, clusters, base_feats, pkb_feats, own_feats, label):
    y = np.asarray(y, float)
    # граница разбиения — квантиль TRAIN_SHARE по дате скоринга: раньше — обучение, позже — тест
    cut = pd.Series(score_date).quantile(TRAIN_SHARE)
    tr = (pd.Series(score_date) <= cut).values
    te = ~tr
    if y[te].sum() < 10 or y[tr].sum() < 10:
        print('Недостаточно «плохих» для проверки вне времени (out-of-time)'); return None
    Xtr, Xte, ytr, yte = X.loc[tr], X.loc[te], y[tr], y[te]
    print(f'{label}: обучающая выборка {tr.sum():,} (доля «плохих» {ytr.mean():.2%}, до {pd.Timestamp(cut).date()}), '
          f'тестовая {te.sum():,} (доля «плохих» {yte.mean():.2%})')
    # отбор на обучающей выборке: базовые признаки (мягкий порог IV, до 30), ПКБ по другим кредиторам и правилам банка,
    # ПКБ по нашим договорам (до 5)
    sel_base, _ = select_on_train(Xtr, ytr, base_feats, max_n=30, iv_min=0.005)
    sel_pkb, iv_tr = select_on_train(Xtr, ytr, pkb_feats)
    sel_own, _ = select_on_train(Xtr, ytr, own_feats, max_n=5)
    # наборы признаков моделей; названия моделей — ключи таблицы результатов (используются и в разделе 17), не переименовывать
    sets = {'A: baseline': sel_base, 'B: + PKB other lenders': sel_base + sel_pkb,
            'C: + PKB own contracts': sel_base + sel_pkb + sel_own}
    # бины и WoE подбираются один раз для всех признаков самой широкой модели C
    enc = woe_fit(Xtr, ytr, sorted(set(sets['C: + PKB own contracts'])))
    res, scores, coefs = [], {}, None
    # для каждой модели: логистическая регрессия на WoE, Джини на обучающей и тестовой выборке,
    # доля выявленных «плохих» в топ-10/20/30% самых рискованных эпизодов теста
    for name, fs in sets.items():
        if not fs:
            continue
        lr = LogisticRegression(C=1.0, max_iter=2000)
        lr.fit(woe_apply(Xtr, enc, fs), ytr)
        p_tr = lr.predict_proba(woe_apply(Xtr, enc, fs))[:, 1]
        p_te = lr.predict_proba(woe_apply(Xte, enc, fs))[:, 1]
        scores[name] = p_te
        res.append({'model': name, 'n_features': len(fs), 'gini_train': gini(ytr, p_tr), 'gini_test': gini(yte, p_te),
                    'capture_top10%': capture(yte, p_te, 0.1), 'capture_top20%': capture(yte, p_te, 0.2),
                    'capture_top30%': capture(yte, p_te, 0.3)})
        if name.startswith('B'):
            # коэффициенты модели B с группой признака и проверкой знака
            coefs = pd.DataFrame({'feature': fs, 'coef': lr.coef_[0]}).set_index('feature')
            coefs['group'] = ['baseline' if f.startswith('bl_') else catalog.loc[f, 'group'] for f in fs]
            coefs['sign_ok'] = coefs['coef'] < 0          # WoE > 0 означает меньший риск, поэтому коэффициенты должны быть отрицательными
    if not res:
        print(f'{label}: на обучающей выборке не отобрано ни одного признака')
        return None
    res = pd.DataFrame(res).set_index('model')
    # ΔДжини моделей B и C относительно модели A на тесте и её бутстрэп 95% ДИ (кластеры — клиенты тестовой выборки)
    cl_te = np.asarray(clusters)[te]
    for name in (res.index[1:] if 'A: baseline' in scores else []):
        lo, mid, hi = boot_delta(yte, scores['A: baseline'], scores[name], cl_te)
        res.loc[name, 'delta_gini_vs_A'] = res.loc[name, 'gini_test'] - res.loc['A: baseline', 'gini_test']
        res.loc[name, 'delta_CI95'] = f'[{lo:+.4f}, {hi:+.4f}]'
    # контрольная проверка на градиентном бустинге по всем признакам без отбора (модели A и B)
    if RUN_HGB and HistGradientBoostingClassifier is not None:
        # матрица для бустинга: числовые признаки как есть, категориальные — в WoE
        def hgb_matrix(Xs, fs):
            cat = [f for f in fs if not pd.api.types.is_numeric_dtype(X[f])]
            num = [f for f in fs if f not in cat]
            return np.column_stack([Xs[num].astype(float).values, woe_apply(Xs, enc_cat, cat)]) if cat else Xs[num].astype(float).values
        cat_feats = [f for f in base_feats if not pd.api.types.is_numeric_dtype(X[f])]
        enc_cat = woe_fit(Xtr, ytr, cat_feats)
        hg = {}
        for name, fs in (('A: baseline', base_feats), ('B: + PKB other lenders', base_feats + pkb_feats)):
            m = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_leaf_nodes=15,
                                               min_samples_leaf=max(50, int(0.002 * tr.sum())),
                                               l2_regularization=1.0, random_state=SEED)
            m.fit(hgb_matrix(Xtr, fs), ytr)
            hg[name] = m.predict_proba(hgb_matrix(Xte, fs))[:, 1]
            res.loc[name, 'gini_test_boosting'] = gini(yte, hg[name])
        lo, mid, hi = boot_delta(yte, hg['A: baseline'], hg['B: + PKB other lenders'], cl_te) if pkb_feats else (NAN,) * 3
        res.loc['B: + PKB other lenders', 'delta_boosting_CI95'] = f'[{lo:+.4f}, {hi:+.4f}]'
    display(res)
    if coefs is not None:
        print('Коэффициенты модели B (в шкале WoE; ожидаемый знак — отрицательный):')
        display(coefs.join(iv_tr[['iv']], how='left') if 'iv' in iv_tr.columns else coefs)
    # кривая доли выявленных «плохих» (gains) моделей A и B на тестовой выборке; диагональ — случайный порядок
    if len(scores) >= 2:
        fig, ax = plt.subplots(figsize=(5.2, 4))
        for (name, s), col in zip(list(scores.items())[:2], (PAL['gray'], PAL['blue'])):
            o = np.argsort(-s, kind='mergesort')
            cum = np.cumsum(yte[o]) / yte.sum()
            ax.plot(np.arange(1, len(o) + 1) / len(o), cum, color=col, label=f'{RU_VALUES.get(name, name)}, Джини {gini(yte, s):.3f}')
        ax.plot([0, 1], [0, 1], color='#c9c8c2', lw=0.8)
        ax.set_xlabel('доля эпизодов, начиная с самых рискованных')
        ax.set_ylabel('доля выявленных «плохих»')
        ax.set_title(f'{label}: доля выявленных «плохих» на выборке вне времени (OOT)')
        ax.legend(loc='lower right')
        plt.tight_layout(); plt.show()
    return res, sets


# Наборы признаков популяции A: базовые (внутренние), ПКБ (другие кредиторы и правила банка), ПКБ по нашим договорам
base_feats_A = list(base_A.columns)
pkb_feats_A = [f for f in F_A.columns if catalog.loc[f, 'group'] in pkb_groups]
own_feats_A = [f for f in F_A.columns if catalog.loc[f, 'group'] == 'pkb_own']
# дата скоринга и ИИН клиента для каждого эпизода (ИИН — кластер для бутстрэпа)
sd_A = pop.set_index('ep_id').loc[X_A.index, 'score_date'].values
iin_A = pop.set_index('ep_id').loc[X_A.index, 'iin'].values
with Timer('Прирост качества, популяция A'):
    inc_A = incremental_value(X_A, y_A, sd_A, iin_A, base_feats_A, pkb_feats_A, own_feats_A, 'Популяция A')

## 15. Дополнительные целевые переменные (таргеты)
Информационная ценность (IV) тех же признаков относительно других определений исхода (бины подбираются заново для каждой
целевой переменной). Дополнительные целевые переменные задаются в `EXTRA_TARGETS` (имя -> порог дней просрочки (DPD)),
например `nocure_16` = 1, если эпизод дошёл до DPD ≥ 16 (не самоизлечился раньше).

In [ ]:
# IV признаков популяции A для каждой дополнительной целевой переменной из EXTRA_TARGETS.
# Учитываются только эпизоды, исход которых уже наблюдаем (непустое значение); нужно не меньше 20 «плохих».
sec = {}
for name in EXTRA_TARGETS:
    yy = pop.set_index('ep_id').loc[X_A.index, name].values
    mk = ~np.isnan(yy)
    if mk.sum() == 0 or np.nansum(yy) < 20:
        print(f'{name}: пока недостаточно эпизодов с наблюдаемым исходом')
        continue
    t, _ = compute_iv(X_A.loc[mk], yy[mk], features_A)
    sec[f'iv_{name}'] = t['iv']
    print(f'{name}: {mk.sum():,} эпизодов, доля «плохих» {np.nanmean(yy[mk]):.2%}')
# сводная таблица: IV по основной целевой переменной (iv_roll<BAD_DPD>) и по дополнительным; показаны признаки ПКБ
if sec:
    iv_targets = iv_A[['group', 'iv']].rename(columns={'iv': f'iv_roll{BAD_DPD}'}).join(pd.DataFrame(sec))
    display(iv_targets[iv_targets['group'].isin(pkb_groups)].head(25))

## 16. Часть B: раннее предупреждение (early warning, EWS) для договоров без просрочки
Те же признаки ПКБ (Первое кредитное бюро, PKB) для договоров, которые на дату среза не находятся в просрочке.
Целевая переменная (таргет): выход на просрочку в течение `EWS_HORIZON` дней. Раздел показывает, могут ли события
мониторинга ПКБ служить поводом для контакта с клиентом ещё до возникновения просрочки.

In [ ]:
# Часть B выполняется, только если популяция B построена и в ней не меньше 20 «плохих».
iv_B, impact_B, inc_B = None, None, None
if pop_B is not None and len(pop_B) and pop_B['target'].sum() >= 20:
    with Timer('Признаки ПКБ, популяция B'):
        F_B, catalog_B = build_pkb_features(pop_B, ev)
    X_B = F_B.join(base_B)
    y_B = pop_B.set_index('ep_id').loc[X_B.index, 'target'].values.astype(float)
    print(f'Популяция B: {len(y_B):,} срезов, доля «плохих» {y_B.mean():.2%}')
    # влияние правил (лифт) и IV признаков — так же, как для популяции A
    impact_B = rule_impact(F_B, y_B, catalog_B)
    display(impact_B.head(25))
    plot_rule_lift(impact_B, f'Популяция B: выход на просрочку в течение {EWS_HORIZON} дн., событие за последние {IMPACT_WINDOW} дн.')
    iv_B, models_B = compute_iv(X_B, y_B, list(X_B.columns), catalog_B)
    display(iv_B[cols_show].head(30))
    plot_iv(iv_B[iv_B['group'].isin(pkb_groups)], 'Популяция B: топ признаков ПКБ по IV')
    # каталог признаков временно дополняется признаками популяции B (он нужен внутри incremental_value),
    # после расчёта восстанавливается исходный каталог
    catalog_main = catalog
    catalog = pd.concat([catalog, catalog_B[~catalog_B.index.isin(catalog.index)]])
    with Timer('Прирост качества, популяция B'):
        inc_B = incremental_value(
            X_B, y_B, pop_B.set_index('ep_id').loc[X_B.index, 'score_date'].values,
            pop_B.set_index('ep_id').loc[X_B.index, 'iin'].values, list(base_B.columns),
            [f for f in F_B.columns if catalog_B.loc[f, 'group'] in pkb_groups],
            [f for f in F_B.columns if catalog_B.loc[f, 'group'] == 'pkb_own'], 'Популяция B')
    catalog = catalog_main
else:
    print('Часть B пропущена')

## 17. Выгрузка результатов и резюме
Результаты записываются в папку `OUTPUT_DIR` (в Excel, если установлен `openpyxl`, иначе — в CSV-файлы).
Резюме ниже формируется автоматически из рассчитанных выше чисел.

Листы выгрузки (в скобках — имя CSV-файла, если `openpyxl` не установлен):
* «IV признаков, популяция A» (`iv_population_A`) — информационная ценность (IV) всех признаков, популяция A;
* «Влияние правил, популяция A» (`rule_impact_A`) — влияние (лифт) правил мониторинга ПКБ (Первое кредитное бюро, PKB);
* «Стабильность, популяция A» (`stability_A`) — понедельная стабильность признаков;
* «Бины признаков, популяция A» (`bins_A`) — таблицы бинов (интервалов) для 80 признаков с наибольшей IV;
* «Когорты, популяция A» (`cohorts_A`) — недельные когорты популяции A;
* «Каталог признаков» (`feature_catalog`) — каталог признаков с описаниями;
* «Сверка с нашими договорами» (`own_contract_matching`) — сопоставление событий ПКБ с нашими договорами по каждому правилу;
* «Прирост качества, популяция A / B» (`incremental_A`, `incremental_B`) — прирост качества (инкрементальный эффект)
  моделей, если он рассчитан;
* «IV по таргетам, популяция A» (`iv_by_target_A`) — IV по дополнительным целевым переменным;
* «IV признаков, популяция B», «Влияние правил, популяция B» (`iv_population_B`, `rule_impact_B`) — то же для
  популяции B (часть B, раннее предупреждение, EWS).

In [ ]:
# Таблицы бинов нескольких признаков, сложенные одна под другой, с колонкой 'feature' (имя признака)
def bins_long(X, y, models, feats):
    out = []
    for f in feats:
        t = bin_table(X[f], y, models[f])
        t.insert(0, 'feature', f)
        out.append(t)
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()


# Листы выгрузки: внутреннее имя (оно же имя CSV-файла) -> таблица
sheets = {'iv_population_A': iv_A, 'rule_impact_A': impact_A, 'stability_A': stab_A,
          'bins_A': bins_long(X_A, y_A, models_A, list(iv_A.head(80).index)),
          'cohorts_A': cohorts, 'feature_catalog': catalog, 'own_contract_matching': own_diag}
if inc_A is not None:
    sheets['incremental_A'] = inc_A[0]
if sec:
    sheets['iv_by_target_A'] = iv_targets
if iv_B is not None:
    sheets.update({'iv_population_B': iv_B, 'rule_impact_B': impact_B})
    if inc_B is not None:
        sheets['incremental_B'] = inc_B[0]
# Русские названия листов Excel (не длиннее 31 символа, без символов []:*?/\); имена файлов не меняются
RU_SHEETS = {'iv_population_A': 'IV признаков, популяция A', 'rule_impact_A': 'Влияние правил, популяция A',
             'stability_A': 'Стабильность, популяция A', 'bins_A': 'Бины признаков, популяция A',
             'cohorts_A': 'Когорты, популяция A', 'feature_catalog': 'Каталог признаков',
             'own_contract_matching': 'Сверка с нашими договорами', 'incremental_A': 'Прирост качества, популяция A',
             'iv_by_target_A': 'IV по таргетам, популяция A', 'iv_population_B': 'IV признаков, популяция B',
             'rule_impact_B': 'Влияние правил, популяция B', 'incremental_B': 'Прирост качества, популяция B'}
try:
    import openpyxl  # noqa: F401
    path = os.path.join(OUTPUT_DIR, 'pkb_monitoring_iv_results.xlsx')
    with pd.ExcelWriter(path) as xw:
        for name, df in sheets.items():
            d = df.copy()
            # колонки с датами записываются как текст
            for c in d.columns:
                if pd.api.types.is_datetime64_any_dtype(d[c]):
                    d[c] = d[c].astype(str)
            # русские подписи колонок и значений (ru_labels), русское имя листа (не длиннее 31 символа)
            ru_labels(d).to_excel(xw, sheet_name=RU_SHEETS.get(name, name)[:31])
    print('Сохранено:', path)
except ImportError:
    # без openpyxl каждая таблица сохраняется в отдельный CSV-файл (с русскими подписями, имя файла — внутреннее)
    for name, df in sheets.items():
        ru_labels(df).to_csv(os.path.join(OUTPUT_DIR, f'{name}.csv'), encoding='utf-8-sig')
    print('CSV-файлы сохранены в папку', OUTPUT_DIR)

# Текстовое резюме: каждая строка формируется из чисел, рассчитанных выше
lines = [f'ДАННЫЕ: {"СИНТЕТИЧЕСКИЕ (демо) — цифры ничего не говорят о реальном портфеле" if DEMO_MODE else "реальные (production)"}',
         f'Использованный период мониторинга ПКБ: {HES_START_TS.date()} .. {HES_END_TS.date()}; данные о днях просрочки (DPD) до {DATA_END_TS.date()}.',
         f'Популяция A: {len(pop):,} выходов в софт-коллекшн с датами скоринга {pop["score_date"].min().date()} .. '
         f'{pop["score_date"].max().date()}; доля перекатов (roll rate) в DPD {BAD_DPD}+ = {BR_A:.2%} («плохих»: {int(y_A.sum()):,}).',
         f'{(F_A["days_since_any_oth"].notna()).mean():.0%} эпизодов имели хотя бы одно событие ПКБ у других кредиторов до даты скоринга.']
# только признаки ПКБ (другие кредиторы и правила банка)
pk = iv_A[iv_A['group'].isin(pkb_groups)]
lines.append(f'Признаков ПКБ: {len(pk)}; IV >= 0.1: {(pk["iv"] >= 0.1).sum()}, 0.02-0.1: {((pk["iv"] >= 0.02) & (pk["iv"] < 0.1)).sum()}, '
             f'ниже 0.02: {(pk["iv"] < 0.02).sum()}.')
lines.append('Самые сильные признаки ПКБ: ' + '; '.join(f'{f} (IV {r.iv:.3f}, {RU_VALUES.get(r.strength, r.strength)})' for f, r in pk.head(5).iterrows()))
# 5 надёжных правил (кроме событий по нашим договорам) с наибольшей IV флага события
top_rules = impact_A[impact_A['reliable'] & (impact_A['group'] != 'pkb_own')].sort_values('iv_flag', ascending=False).head(5)
lines.append('Правила с наибольшим влиянием (событие за последние %d дн.): ' % IMPACT_WINDOW + '; '.join(
    f'{r.rule_key} (лифт {r.lift:.2f}, флаг у {r.share_flagged:.1%} эпизодов)' for r in top_rules.itertuples()))
if inc_A is not None:
    r = inc_A[0]
    b = r.loc['B: + PKB other lenders'] if 'B: + PKB other lenders' in r.index else None
    if b is not None:
        lines.append(f'Джини вне времени (OOT): базовая модель {r.loc["A: baseline", "gini_test"]:.3f} -> с признаками ПКБ {b["gini_test"]:.3f} '
                     f'(прирост {b["delta_gini_vs_A"]:+.3f}, 95% ДИ {b["delta_CI95"]}).')
# подозрительно высокая IV (>= 0.5) — признак нужно проверить на утечку данных
susp = pk[pk['iv'] >= 0.5]
if len(susp):
    lines.append('Проверить на утечку данных (leakage), IV >= 0.5: ' + ', '.join(susp.index[:10]))
if iv_B is not None:
    pb = iv_B[iv_B['group'].isin(pkb_groups)]
    lines.append(f'Часть B (раннее предупреждение, EWS): лучший признак ПКБ {pb.index[0]} (IV {pb["iv"].iloc[0]:.3f}).')
lines.append(f'Оговорка: история мониторинга короткая ({(HES_END_TS - HES_START_TS).days} дн.), поэтому у {max(WINDOWS)}-дневных '
             'признаков в первые недели окно наблюдения усечено, а стабильность проверяется лишь на нескольких недельных когортах.')
print('\n'.join('- ' + s for s in lines))